# Documentation
**Author:** Spencer Ressel

**Created:** February 20th, 2024

***

This notebook analyzes aqua-planet simulation data from the CAM6 model run by Mu-Ting Chien.
For simulation details, see: https://doi.org/10.1029/2024MS004378

In particular, regression analysis is performed on variable data to understand the structure of the Madden-Julian Oscillation in the simulation data

### This file requires other files in the aquaplanet_analysis folder and in the auxiliary_functions folder
 - config.py
 - load_aquaplanet_data.py

***

# Imports

In [1]:
%load_ext autoreload
%autoreload 2

# Aquaplanet analysis config file
import config
import coords

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

logger.info("Loading Imports...")
# File management
import glob
import os
import sys
from datetime import datetime, timedelta
import copy
import cftime

# Data anaylsis
import numpy as np
import scipy
import xarray as xr
import xeofs as xe
xr.set_options(keep_attrs=True)
import scipy.signal as signal
from scipy.stats import t

# Plotting
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
from cartopy import util as cutil
# from plotting_utils import get_plotting_attributes
plt.rcParams['mathtext.fontset'] = 'dejavusans'
import string

# Auxiliary functions
# sys.path.insert(0, "/glade/u/home/sressel/thesis-work/python/auxiliary_functions/")
from load_aquaplanet_data import load_multi_experiment_processed_data
from auxiliary_functions import xarray_utils
from auxiliary_functions.plotting_utils import bmh_colors, tick_labeller, set_plot_mode, get_figsize

logger.info("Imports loaded")

2026-06-03 09:46:42,762 [INFO] Loading Imports...
2026-06-03 09:46:45,356 [INFO] Imports loaded


# Load data

In [9]:
variables_to_load = [
    "Precipitation",
    # "Outgoing Longwave Radiation",
    # "Zonal Wind",
    "Meridional Wind",
    "Vertical Wind",
    # "Temperature",
    "Moisture",
    # "Relative Humidity",
    # "Geopotential Height",
    # "Longwave Heating Rate",
    # "Shortwave Heating Rate",
    # "Latent Heat Flux",
    # "Sensible Heat Flux",
    # "Surface Pressure",
    # "Moist Static Energy",
    # "Column Moist Static Energy",
    "Column Water Vapor",
    # "Column Temperature",
    # "Column Longwave Heating",
    # "Column Shortwave Heating",
    # "Potential Temperature",
    # "Saturation Specific Humidity",
    # "Column Relative Humidity",
    # "Diabatic Heating",
    # "Chikira alpha",
    # "Zonal Moisture Advection",
    # "Meridional Moisture Advection",
    # "Vertical Moisture Advection",
    # 'Moisture Tendency',
    # 'Zonal Advection',
    # 'Meridional Advection',
    # 'Vertical Advection',
    # 'Evaporation',
    # 'Residual'
    # "Upper Level Moisture",
    # "Lower Level Moisture"
]

# Load pre-processed data

## Subset data

In [10]:
if not 'multi_experiment_variables_subset' in locals() or reload_variables:
    multi_experiment_variables_subset = load_multi_experiment_processed_data(
        variables_to_load,
        'subset'
    )
else:
    multi_experiment_variables_subset = load_multi_experiment_processed_data(
        variables_to_load,
        'subset',
        multi_experiment_variables_subset,
        False
    )

2026-06-03 09:50:44,838 [INFO] Loading subset data
2026-06-03 09:50:44,840 [INFO] (1/5) Precipitation...
2026-06-03 09:50:44,841 [INFO]     Experiment: -4K...
2026-06-03 09:50:44,950 [INFO]     Experiment: 0K...
2026-06-03 09:50:45,050 [INFO]     Experiment: 4K...
2026-06-03 09:50:45,751 [INFO] (2/5) Meridional Wind...
2026-06-03 09:50:45,753 [INFO]     Experiment: -4K...
2026-06-03 09:50:45,802 [INFO]     Experiment: 0K...


2026-06-03 09:50:45,912 [INFO]     Experiment: 4K...
2026-06-03 09:50:58,185 [INFO] (3/5) Vertical Wind...
2026-06-03 09:50:58,186 [INFO]     Experiment: -4K...
2026-06-03 09:50:58,283 [INFO]     Experiment: 0K...
2026-06-03 09:50:58,383 [INFO]     Experiment: 4K...
2026-06-03 09:51:12,457 [INFO] (4/5) Moisture...
2026-06-03 09:51:12,459 [INFO]     Experiment: -4K...
2026-06-03 09:51:12,551 [INFO]     Experiment: 0K...
2026-06-03 09:51:12,605 [INFO]     Experiment: 4K...
2026-06-03 09:51:22,019 [INFO] (5/5) Column Water Vapor...
2026-06-03 09:51:22,040 [INFO]     Experiment: -4K...
2026-06-03 09:51:22,133 [INFO]     Experiment: 0K...
2026-06-03 09:51:22,222 [INFO]     Experiment: 4K...
2026-06-03 09:51:23,792 [INFO] Finished


## Budget Variables

In [ ]:
budget_to_load = 'moisture'
data_source_to_load = 'daily_model-level'
budget_data_location = f"{config.DATA_DIRECTORY}/{budget_to_load}_budget_terms/{data_source_to_load}_data"
multi_experiment_budget_variables = {}

budget_variables_to_load = [
            'Moisture',
            'Moisture Tendency',
            'Zonal Advection',
            'Meridional Advection',
            'Vertical Advection',
            'Evaporation',
            'Precipitation',
            'Residual'
        ]

for index, variable_name in enumerate(budget_variables_to_load):
    data = xr.open_dataset(f"{budget_data_location}/multi_experiment_{variable_name.lower().replace(' ', '_')}.nc")
    for variable_data in data.data_vars.values():
        print(f"{f'({index+1}/{len(budget_variables_to_load)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
        multi_experiment_budget_variables[variable_name] = variable_data
        print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Time-filtered

In [11]:
if not 'multi_experiment_variables_filtered' in locals():
    multi_experiment_variables_filtered = load_multi_experiment_processed_data(
        ['Vertical Wind'],# 'Upper Level Moisture', 'Lower Level Moisture'],
        'filtered'
    )
else:
    multi_experiment_variables_filtered = load_multi_experiment_processed_data(
        ['Vertical Wind'],# 'Upper Level Moisture', 'Lower Level Moisture'],
        'filtered',
        multi_experiment_variables_filtered,
        False
    )

2026-06-03 09:51:31,969 [INFO] Loading filtered data
2026-06-03 09:51:31,971 [INFO] (1/1) Vertical Wind...
2026-06-03 09:51:31,972 [INFO]     Experiment: -4K...
2026-06-03 09:51:32,022 [INFO]     Experiment: 0K...
2026-06-03 09:51:32,111 [INFO]     Experiment: 4K...
2026-06-03 09:51:39,407 [INFO] Finished


## MJO-filtered

In [12]:
if not 'multi_experiment_variables_mjo_filtered' in locals():
    multi_experiment_variables_mjo_filtered = load_multi_experiment_processed_data(
        ['Precipitation'],# 'Upper Level Moisture', 'Lower Level Moisture'],
        'mjo_filtered'
    )
else:
    multi_experiment_variables_mjo_filtered = load_multi_experiment_processed_data(
        ['Precipitation'],# 'Upper Level Moisture', 'Lower Level Moisture'],
        'mjo_filtered',
        multi_experiment_variables_mjo_filtered,
        False
    )

2026-06-03 09:51:44,379 [INFO] Loading mjo_filtered data
2026-06-03 09:51:44,380 [INFO] (1/1) Precipitation...
2026-06-03 09:51:44,381 [INFO]     Experiment: -4K...
2026-06-03 09:51:44,638 [INFO]     Experiment: 0K...
2026-06-03 09:51:44,733 [INFO]     Experiment: 4K...
2026-06-03 09:51:45,686 [INFO] Finished


## Pre-processed Regression data

In [4]:
data_type = 'regressed'
rescaled = False
# reference_variable = 'QU-QL'
reference_variable = 'PRCP'
regression_wave_type = 'MJO'

print(f"{f'Loading {data_type.title()} data':^{config.SEP_WIDTH}}")
# print(f"{'='*config.SEP_WIDTH}")

reload_regressed = True
if not 'multi_experiment_variables_regressed' in globals() or reload_regressed:
    multi_experiment_variables_regressed = {}
if not 'multi_experiment_significant_indices' in globals() or reload_regressed:
    multi_experiment_significant_indices = {}

if reference_variable == 'PRCP':
    for index, variable_name in enumerate(variables_to_load):
        # Print the variable name
        print(f"{'='*config.SEP_WIDTH}")
        print(f"{f'({index+1}/{len(variables_to_load)}) {variable_name}...':<{config.SEP_WIDTH}}")
        print(f"{'-'*config.SEP_WIDTH}")

        # For each variable, loop over the experiments and find/load the data
        print("Loading data...")
        file_location = rf"{config.DATA_DIRECTORY}/multi_experiment_variables_{data_type}"
        regressed_files = glob.glob(
            f"{file_location}/multi_location_multi_experiment"
            + f"_{('rescaled_' if rescaled and data_type == 'regressed' else '')}"
            + f"{variable_name.lower().replace(' ', '_')}"
            + f"_{regression_wave_type}-{reference_variable}.nc"
        )

        # Does the variable have data?
        if regressed_files:
            # Append it to the list
            for index, file in enumerate(regressed_files):
                regression_data = xr.open_dataset(file)
                print(f"{f'    Coefficients...':<{config.SEP_WIDTH-1}}", end="")
                multi_experiment_variables_regressed[variable_name] = regression_data[f"{variable_name} - Coefficients"]
                multi_experiment_variables_regressed[variable_name].name = variable_name
                print(rf"{'✔':>1}")

                print(f"{f'    Significant Indices...':<{config.SEP_WIDTH-1}}", end="")
                multi_experiment_significant_indices[variable_name] = regression_data[f"{variable_name} - Significance"]
                multi_experiment_significant_indices[variable_name].name = variable_name
                print(rf"{'✔':>1}")

        else:
            # Move on to the next variable
            print(rf"{'✘':>1}")
            print(f"    No data")

elif reference_variable == 'QU-QL':
    for index, variable_name in enumerate(variables_to_load):
        # Print the variable name
        print(f"{'='*config.SEP_WIDTH}")
        print(f"{f'({index+1}/{len(variables_to_load)}) {variable_name}...':<{config.SEP_WIDTH}}")
        print(f"{'-'*config.SEP_WIDTH}")

        multi_experiment_variables_regressed[variable_name] = {}
        multi_experiment_significant_indices[variable_name] = {}
        for reference_variable in ['QU', 'QL']:

            # For each variable, loop over the experiments and find/load the data
            print(f"Loading {reference_variable} data...")
            file_location = rf"{data_directory}/multi_experiment_variables_{data_type}"
            regressed_files = glob.glob(
                f"{file_location}/multi_experiment_{('rescaled_' if data_type == 'regressed' and rescaled else '')}{variable_name.lower().replace(' ', '_')}"
                + f"_{regression_wave_type}-{reference_variable}.nc"
            )

            # Does the variable have data?
            if regressed_files:
                # Append it to the list
                for index, file in enumerate(regressed_files):
                    regression_data = xr.open_dataset(file)
                    print(f"{f'    Coefficients...':<{config.SEP_WIDTH-1}}", end="")
                    multi_experiment_variables_regressed[variable_name][reference_variable] = regression_data[f"{variable_name} - Coefficients"]
                    multi_experiment_variables_regressed[variable_name][reference_variable].name = variable_name
                    print(rf"{'✔':>1}")

                    print(f"{f'    Significant Indices...':<{config.SEP_WIDTH-1}}", end="")
                    multi_experiment_significant_indices[variable_name][reference_variable] = regression_data[f"{variable_name} - Significance"]
                    multi_experiment_significant_indices[variable_name][reference_variable].name = variable_name
                    print(rf"{'✔':>1}")

            else:
                # Move on to the next variable
                print(rf"{'✘':>1}")
                print(f"    No data")

        multi_experiment_variables_regressed[variable_name] = xr.concat(
            [multi_experiment_variables_regressed[variable_name][reference_variable] for reference_variable in ['QU', 'QL']],
            dim=xr.DataArray(data=['upper', 'lower'], dims='level', coords={'level':['upper', 'lower']})
        )

        multi_experiment_significant_indices[variable_name] = xr.concat(
            [multi_experiment_significant_indices[variable_name][reference_variable] for reference_variable in ['QU', 'QL']],
            dim=xr.DataArray(data=['upper', 'lower'], dims='level', coords={'level':['upper', 'lower']})
        )

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

         Loading Regressed data         
(1/4) Precipitation...                  
----------------------------------------
Loading data...
    Coefficients...                    ✔
    Significant Indices...             ✔
(2/4) Meridional Wind...                
----------------------------------------
Loading data...
    Coefficients...                    ✔
    Significant Indices...             ✔
(3/4) Moisture...                       
----------------------------------------
Loading data...
    Coefficients...                    ✔
    Significant Indices...             ✔
(4/4) Column Water Vapor...             
----------------------------------------
Loading data...
    Coefficients...                    ✔
    Significant Indices...             ✔
Finished


In [6]:
multi_experiment_variables_regressed['Column Water Vapor'].max(dim=['lat', 'lon'])

<xarray.DataArray 'Column Water Vapor' (experiment: 3)> Size: 24B
array([0.44744922, 0.93564563, 2.76211649])
Coordinates:
  * experiment  (experiment) <U3 36B '-4K' '0K' '4K'
Attributes:
    description:  Column water vapor from 1000 hPa to 100 hPa
    units:        mm
    file_id:      CWV
    short_name:   $\langle$Q$\rangle$
    subset:       True

In [8]:
multi_experiment_variables_regressed['Meridional Wind'].sel(plev=850).max(dim=['lat', 'lon'])

<xarray.DataArray 'Meridional Wind' (experiment: 3)> Size: 24B
array([0.1455435 , 0.23818546, 0.53390983])
Coordinates:
    plev        int64 8B 850
  * experiment  (experiment) <U3 36B '-4K' '0K' '4K'
Attributes:
    description:  Meridional Wind
    units:        m s$^{-1}$
    file_id:      V
    short_name:   V
    subset:       True

In [ ]:
zonal_wind_gradient = (
    (180/np.pi)
    * multi_experiment_variables_regressed['Zonal Wind'].differentiate('lon')
    / (config.EARTH_RADIUS*np.cos(np.deg2rad(multi_experiment_variables_regressed['Zonal Wind'].lat)))
)

meridional_wind_gradient = (
    (180/np.pi)
    * multi_experiment_variables_regressed['Meridional Wind'].differentiate('lat')
    / config.EARTH_RADIUS
)

divergence = zonal_wind_gradient + meridional_wind_gradient

In [ ]:
divergence.sel(lat=slice(-15,15), lon=slice(175,185)).mean(dim=['lat', 'lon']).plot(y='plev', yincrease=False, hue='experiment')

In [ ]:
plt.style.use('bmh')
divergence.plot(y='plev', yincrease=False, hue='experiment')

In [ ]:
[fig, ax] = plt.subplots(1, 3, figsize=(16,4))
fig.suptitle('MJO Moisture gradient (colors) and mean winds (contours), lons 160°-200°', y=0.95, size=14)

for axis, experiment in zip(ax, coords.experiments.values):
    axis.contourf(
        multi_experiment_variables_regressed['Moisture'].lat,
        multi_experiment_variables_regressed['Moisture'].plev,
        multi_experiment_variables_regressed['Moisture'].sel(experiment=experiment, lon=slice(160,200)).mean(dim='lon').differentiate('lat').T,
        cmap='coolwarm',
        levels=21
    )
    axis.contour(
        multi_experiment_variables_subset['Meridional Wind'].lat,
        multi_experiment_variables_subset['Meridional Wind'].plev,
        multi_experiment_variables_subset['Meridional Wind'].sel(experiment=experiment, lon=slice(160,200)).mean(dim=['time', 'lon']).T,
        colors='k',
        levels=np.linspace(-0.8,0.8,11),
        linewidths=1
    )
    axis.contour(
        multi_experiment_variables_subset['Meridional Wind'].lat,
        multi_experiment_variables_subset['Meridional Wind'].plev,
        multi_experiment_variables_subset['Meridional Wind'].sel(experiment=experiment, lon=slice(160,200)).mean(dim=['time', 'lon']).T,
        colors='k',
        levels=[0],
        linewidths=1.5
        )

    axis.invert_yaxis()
    axis.set_xlabel('lat')
    axis.set_ylabel('plev')
plt.show()

In [ ]:
 multi_experiment_variables_subset['Meridional Wind'].sel(experiment=experiment, lon=slice(160,200)).mean(dim=['time', 'lon']).min()

In [ ]:
prod = multi_experiment_variables_regressed['Moisture'].sel(lon=slice(160,200)).mean(dim='lon').differentiate('lat') * multi_experiment_variables_subset['Meridional Wind'].sel(lon=slice(160,200)).mean(dim=['time', 'lon'])

# (
#     prod/np.abs(prod).max(dim=['lat', 'plev'])
# ).plot.contourf(x='lat', y='plev', yincrease=False, col='experiment', levels=21, figsize=(16,6))

[fig, ax] = plt.subplots(1, 3, figsize=(16,4))
# fig.suptitle('MJO Moisture (colors) and mean winds (contours), lons 160°-200°', y=0.95, size=14)

for axis, experiment in zip(ax, coords.experiments.values):
    axis.contourf(
        prod.lat,
        prod.plev,
        prod.sel(experiment=experiment).T,
        cmap='coolwarm',
        levels=21
    )
    axis.contour(
        multi_experiment_variables_regressed['Moisture'].lat,
        multi_experiment_variables_regressed['Moisture'].plev,
        multi_experiment_variables_regressed['Moisture'].sel(experiment=experiment, lon=slice(160,200)).mean(dim=['lon']).T,
        colors='k',
        levels=11,
        linewidths=0.5
    )
    axis.contour(
            multi_experiment_variables_regressed['Moisture'].lat,
            multi_experiment_variables_regressed['Moisture'].plev,
            multi_experiment_variables_regressed['Moisture'].sel(experiment=experiment, lon=slice(160,200)).mean(dim=['lon']).T,
            colors='k',
            levels=[0],
            linewidths=1
        )

    axis.invert_yaxis()
    axis.set_xlabel('lat')
    axis.set_ylabel('plev')
plt.show()

In [ ]:
plt.style.use('bmh')
plt.rcParams.update({'font.size':16})
# [fig, ax] = plt.subplots(1, 2, figsize=(16,6))

fig = plt.figure(figsize=(12, 8))
gs = GridSpec(1, 1, figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95)
ax = np.array([fig.add_subplot(gs[i]) for i in range(1)])

# multi_experiment_variables_mjo_filtered['Precipitation'].sel(lat=slice(-5,5)).mean(dim=['lat', 'lon']).std(dim='time').plot(ax=ax[0])
# multi_experiment_variables_mjo_filtered['Precipitation'].sel(lat=slice(-10,10)).mean(dim=['lat', 'lon']).std(dim='time').plot(ax=ax[0])

ax[0].spines['left'].set_linewidth(3)
ax[0].spines['left'].set_linestyle('-')
ax[0].spines['left'].set_color('goldenrod')

experiment_mean_precipitation.plot(ax=ax[0], color='goldenrod', ls='-', label='Mean-state', marker='^')
experiment_mean_precipitation.plot.scatter(ax=ax[0], c=[bmh_colors(1), bmh_colors(2), bmh_colors(3)], marker='^', s=100, zorder=100)

experiment_max_mjo_precipitation.plot(ax=ax[0], color='goldenrod',  label='MJO-regressed', marker='o', ls='--')
experiment_max_mjo_precipitation.plot.scatter(ax=ax[0], c=[bmh_colors(1), bmh_colors(2), bmh_colors(3)], marker='o', s=100, zorder=100)

ax[0].legend(loc='upper left')
ax[0].set_ylim(0, 6.5)

ax1 = ax[0].twinx()
# ax1.spines['left'].set_color(bmh_colors(1))
# ax1.spines['left'].set_linewidth(2)
ax1.spines['right'].set_linewidth(3)
ax1.spines['right'].set_linestyle('-')
ax1.spines['right'].set_color(bmh_colors(4))
ax1.grid(False)

experiment_mean_surface_temperature.plot(ax=ax1, color=bmh_colors(4), ls='-', marker='s')
experiment_mean_surface_temperature.plot.scatter(ax=ax1, x='experiment', c=[bmh_colors(1), bmh_colors(2), bmh_colors(3)], marker='s', s=100, zorder=100)

for axis in ax.ravel():
    axis.set_xticks(ticks=['-4K', '0K', '4K'], labels=[config.EXPERIMENT_DISPLAY_NAMES[experiment] for experiment in coords.experiment.values])
    axis.set_xlabel('Experiment')
# ax1.set_title('Changes in Precipitation \n & Surface Temperature with SST Warming', fontsize=20)
ax1.set_title('')
plt.show()

# Calculate regressions

## Define Reference Timeseries

## Regress variables

### Multiple Regions

In [ ]:
variable_data = multi_experiment_variables_subset['Moisture']

longitude_centers = np.arange(0, 360, 60)
longitude_coords = xr.DataArray(longitude_centers, dims="region", coords={'region':longitude_centers}, name="region_lon")
distance_to_center = xr.ufuncs.mod(coords.longitude - longitude_coords, 360)
distance_to_center = xr.where(distance_to_center > 180, 360 - distance_to_center, distance_to_center)
lon_mask = distance_to_center <= 5
lat_mask = (coords.latitude >= -5) & (coords.latitude <= 5)
mask = lon_mask & lat_mask

masked = multi_experiment_variables_mjo_filtered['Precipitation'].where(mask)
mean_precipitation_by_region = masked.mean(dim=["lat", "lon"])

# Initialize the dictionary to hold the regressed values
standardized_reference_timeseries = standardize_data(mean_precipitation_by_region, unit_variance=True, dim='time')

N = len(standardized_reference_timeseries.time)
t_critical = t.ppf(0.95, df=N-2)

# Average over regions
# Step 1: compute wrapped distance between each region center and every grid lon
distance_degrees = (coords.longitude.values - longitude_coords.values[:, None] + 540) % 360 - 180

# Step 2: choose nearest lon index
region_center_idx = abs(distance_degrees).argmin(axis=1)

# Step 3: choose target index (I recommend mid-grid)
target_idx = list(coords.longitude.lon.values).index(180)

# Step 4: compute shift amount
shift = target_idx - region_center_idx
variable_data_anomalies = standardize_data(variable_data, unit_variance=False, dim='time')


variable_regressed_by_region = xr.dot(
    standardized_reference_timeseries,
    variable_data_anomalies,
    dim='time'
) / len(standardized_reference_timeseries.time)

variable_regressed_with_shift = variable_regressed_by_region.sel(plev=850).assign_coords(
    shift_amount=("region", shift)
).groupby("region").apply(
    lambda x: x.roll(lon=int(x.shift_amount), roll_coords=False)
)

In [ ]:
masked.sel(experiment='4K').isel(time=0).plot.contourf(x='lon', y='lat', col='region', levels=21)

In [ ]:
variable_regressed_by_region.sel(experiment='4K', plev=850).plot.contourf(x='lon', y='lat', col='region', levels=21)

In [ ]:
variable_regressed_with_shift.sel(experiment='4K').plot.contourf(x='lon', y='lat', col='region', levels=21)

In [ ]:
recalculate_regressions = True
save_regressed_variables = False
reference_variable = 'Precipitation'
data_type = 'regressed'

longitude_centers = np.arange(0, 360, 60)
longitude_coords = xr.DataArray(longitude_centers, dims="region", coords={'region':longitude_centers}, name="region_lon")
distance_to_center = xr.ufuncs.mod(coords.longitude - longitude_coords, 360)
distance_to_center = xr.where(distance_to_center > 180, 360 - distance_to_center, distance_to_center)
lon_mask = distance_to_center <= 5
lat_mask = (coords.latitude >= -5) & (coords.latitude <= 5)
mask = lon_mask & lat_mask

masked = multi_experiment_variables_mjo_filtered['Precipitation'].where(mask)
mean_precipitation_by_region = masked.mean(dim=["lat", "lon"])

multi_experiment_variables_regressed_by_region = {}

# Initialize the dictionary to hold the regressed values
standardized_reference_timeseries = standardize_data(mean_precipitation_by_region, unit_variance=True, dim='time')

N = len(standardized_reference_timeseries.time)
t_critical = t.ppf(0.95, df=N-2)

# Average over regions
# Step 1: compute wrapped distance between each region center and every grid lon
distance_degrees = (coords.longitude.values - longitude_coords.values[:, None] + 540) % 360 - 180

# Step 2: choose nearest lon index
region_center_idx = abs(distance_degrees).argmin(axis=1)

# Step 3: choose target index (I recommend mid-grid)
target_idx = list(coords.longitude.lon.values).index(180)

# Step 4: compute shift amount
shift = target_idx - region_center_idx

print(f"{'Regressions':^{config.SEP_WIDTH}}")
print(f"{f'Regression Variable: {reference_variable}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

if 'multi_experiment_variables_regressed' not in locals() or recalculate_regressions:
    multi_experiment_variables_regressed = {}
    significant_regression_indices = {}
    multi_experiment_variables_std = {}

for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_subset.items()):

    print(f'{f"({index+1}/{len(multi_experiment_variables_subset)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")
    if not recalculate_regressions and variable_name in multi_experiment_variables_regressed:
        print(rf"{'✔':>1}")
        continue

    # Remove the time mean of the variables to be regressed
    variable_data_anomalies = standardize_data(variable_data, unit_variance=False, dim='time')

    if 'plev' not in variable_data.coords:
        variable_regressed_by_region = xr.dot(
            standardized_reference_timeseries,
            variable_data_anomalies,
            dim='time'
        ) / len(standardized_reference_timeseries.time)

        variable_regressed_with_shift = variable_regressed_by_region.assign_coords(
            shift_amount=("region", shift)
        ).groupby("region").apply(
            lambda x: x.roll(lon=int(x.shift_amount), roll_coords=False)
        )
        multi_experiment_variables_regressed[variable_name] = variable_regressed_with_shift.mean(dim='region')

        predicted_value = variable_regressed_with_shift * standardized_reference_timeseries
        residual_sum_of_squares = ((variable_data_anomalies - predicted_value) ** 2).sum(dim='time')
        residual_variance = residual_sum_of_squares / (N - 2)
        standard_error_slope = np.sqrt(residual_variance / (N - 1))
        t_statistic = variable_regressed_with_shift / standard_error_slope

        combined_t_statistic = -2*np.log(np.abs(t_statistic)).sum(dim='region')
        significant_regression_indices[variable_name] = np.abs(combined_t_statistic) > t_critical

        del variable_regressed_by_region, variable_regressed_with_shift

    else:
        multi_experiment_variables_regressed[variable_name] = xr.zeros_like(
            variable_data.isel(time=0, drop=True)
        ).transpose(..., 'plev')
        significant_regression_indices[variable_name] = xr.zeros_like(
            variable_data.isel(time=0, drop=True)
        ).transpose(..., 'plev')
        for (level_index, plev) in enumerate(variable_data.plev):
            if (plev % 250 == 0):
                print(f"{("\n" if plev == 250 else "")}{plev.values} hPa")

            variable_regressed_by_region_by_level = xr.dot(
                standardized_reference_timeseries,
                variable_data_anomalies.sel(plev=plev),
                dim='time'
            ) / len(standardized_reference_timeseries.time)

            variable_regressed_with_shift = variable_regressed_by_region_by_level.assign_coords(
                shift_amount=("region", shift)
            ).groupby("region").apply(
                lambda x: x.roll(lon=int(x.shift_amount), roll_coords=False)
            )
            multi_experiment_variables_regressed[variable_name][..., level_index] = variable_regressed_with_shift.mean(dim='region')

            predicted_value = variable_regressed_with_shift * standardized_reference_timeseries
            residual_sum_of_squares = ((variable_data_anomalies.sel(plev=plev) - predicted_value) ** 2).sum(dim='time')
            residual_variance = residual_sum_of_squares / (N - 2)
            standard_error_slope = np.sqrt(residual_variance / (N - 1))
            t_statistic = variable_regressed_with_shift / standard_error_slope

            combined_t_statistic = -2*np.log(np.abs(t_statistic)).sum(dim='region')
            significant_regression_indices[variable_name][..., level_index] = np.abs(combined_t_statistic) > t_critical

            del variable_regressed_by_region_by_level, variable_regressed_with_shift

    print(rf"{'✔':>1}")

    multi_experiment_variables_regressed[variable_name].attrs = variable_data_anomalies.attrs
    multi_experiment_variables_regressed[variable_name].name = variable_name

    # Calculate the statstical significance of the regression coefficients
    # predicted_value = multi_experiment_variables_regressed[variable_name] * standardized_reference_timeseries
    # residual_sum_of_squares = ((variable_data_anomalies - predicted_value) ** 2).sum(dim='time')
    # residual_variance = residual_sum_of_squares / (N - 2)
    # standard_error_slope = np.sqrt(residual_variance / (N - 1))
    # t_statistic = multi_experiment_variables_regressed[variable_name] / standard_error_slope
    # t_statistic = multi_experiment_variables_regressed[variable_name]/multi_experiment_variables_std[variable_name]/np.sqrt(12)
    # t_critical = t.ppf(0.95, df=11)
    # significant_regression_indices[variable_name] = np.abs(t_statistic) > t_critical
    # print(rf"{'✔':>1}")

print(f"{'-'*config.SEP_WIDTH}")

if save_regressed_variables:
    print(f"{'Saving regressed variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_regressed.items()):
        print(f"{f'({index+1}/{len(multi_experiment_variables_regressed)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
        multi_experiment_variables_regressed[variable_name].name = variable_name
        significant_regression_indices[variable_name].name = variable_name

        filename = (
            f"{data_directory}/multi_experiment_variables_regressed/"
            + f"multi_location_multi_experiment_{variable_name.lower().replace(' ', '_')}_"
            # + f"MJO-{rescaled_multi_experiment_variables_regressed[reference_variable].attrs['file_id']}.nc"
            + f"MJO-{multi_experiment_variables_mjo_filtered[reference_variable].attrs['file_id']}.nc"
        )
        combined_data = xr.Dataset(
            {
                f"{variable_name} - Coefficients": multi_experiment_variables_regressed[variable_name],
                f"{variable_name} - Significance": significant_regression_indices[variable_name]
            }
        )
        if os.path.exists(filename):
            # Prompt user for confirmation
            # response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            response = 'y'
            if response == 'y':
                os.remove(filename)  # Delete the existing file

                # combined_data["name"] = variable_data.name
                combined_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            combined_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving regressed variables':<{config.SEP_WIDTH}}")


# print("Re-scaling regressed timeseries")
# rescaled_multi_experiment_variables_regressed = {}
# rescaled_multi_experiment_variables_regressed = copy.deepcopy(multi_experiment_variables_regressed)
# print(f"{'-'*config.SEP_WIDTH}")

# reference_variable = 'Precipitation'
# rescaling_constant = {}
# rescaling_constant['Outgoing Longwave Radiation'] = -40
# rescaling_constant['Precipitation'] = 15
# rescaling_constant['Upper Level Moisture'] = 10
# rescaling_constant['Lower Level Moisture'] = 10

# for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_regressed.items()):
#     print(f'{f"({index+1}/{len(multi_experiment_variables_regressed)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")

#     for ((experiment_index, experiment), latitude, longitude) in zip(enumerate(coords.experiments.values), reference_latitude, reference_longitude):
#         # print(f'{f"{experiment}...":<{config.SEP_WIDTH-1}}', end="")
#         rescaling_factor = (
#             rescaling_constant[reference_variable]
#             / multi_experiment_variables_regressed[reference_variable].sel(experiment=experiment).sel(lat=latitude, lon=longitude, method='nearest')
#         )
#         rescaled_multi_experiment_variables_regressed[variable_name][experiment_index] = (
#             rescaling_factor.copy(deep=True)
#             * multi_experiment_variables_regressed[variable_name].sel(experiment=experiment)
#         )
#         rescaled_multi_experiment_variables_regressed[variable_name].attrs['rescaled'] = 'True'
#     print(rf"{'✔':>1}")
# print(f"{'-'*config.SEP_WIDTH}")

# multi_experiment_variables_regressed = {}
# for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_regressed_by_region.items()):

#     print(f'{f"({index+1}/{len(multi_experiment_variables_regressed_by_region)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")
#     variable_data_with_shift = variable_data.assign_coords(shift_amount=("region", shift))
#     multi_experiment_variables_regressed[variable_name] = variable_data_with_shift.groupby("region").apply(
#         lambda x: x.roll(lon=int(x.shift_amount), roll_coords=False)
#     ).mean(dim='region')
#     print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
multi_experiment_significant_indices = significant_regression_indices

In [ ]:
underline = "\033[4m"
reset = "\033[0m"

save_regressed_variables = False
reference_variable = 'Precipitation'
# reference_variable = 'Lower Level Moisture'
data_type = 'regressed'

# Initialize the dictionary to hold the regressed values
multi_experiment_variables_regressed = {}

# # Identify the location of the reference time series
reference_latitude = (
    multi_experiment_variables_mjo_filtered['Precipitation']
).var(dim='time').mean(dim='lon').idxmax(dim='lat')
reference_longitude = 180*xr.ones_like(reference_latitude)

reference_latitude = xr.zeros_like(reference_latitude)
reference_longitude = 180*xr.ones_like(reference_latitude)

# selected_data = [
#     ds.sel(
#         lon=reference_longitude.sel(experiment=experiment),
#         lat=reference_latitude.sel(experiment=experiment),
#         method="nearest"
#     )
#     for experiment, ds in zip(experiments_list, multi_experiment_variables_mjo_filtered[reference_variable])
#     # for experiment, ds in zip(experiments_list, mjo_filtered_moisture.sel(plev=slice(100,975)).integrate('plev'))
# ]

# # Concatenate along the 'experiment' dimension
# reference_timeseries = xr.concat(selected_data, dim="experiment")
# reference_longitude = 180
# reference_latitude = 0
reference_timeseries = multi_experiment_variables_mjo_filtered[reference_variable].sel(lat=0, lon=180, method='nearest')
# longitude_bounds = (175, 185)
# latitude_bounds = (-5, 5)
# reference_timeseries = multi_experiment_variables_mjo_filtered[reference_variable].sel(
#         lat=slice(*coords.latitude_bounds),
#         lon=slice(*longitude_bounds)
# ).mean(dim=['lat', 'lon'])
standardized_reference_timeseries = standardize_data(reference_timeseries, unit_variance=True, dim='time')

N = len(standardized_reference_timeseries.time)
t_critical = t.ppf(0.95, df=N-2)

significant_regression_indices = {}

print(f"{'Regressions':^{config.SEP_WIDTH}}")
print(f"{f'Regression Variable: {reference_variable}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

# Take a meridional mean over the specified domain
meridional_mean_data = {}
for variable_data in [
    multi_experiment_variables_subset['Meridional Moisture Advection'],
    multi_experiment_variables_subset['Moisture'],
    multi_experiment_variables_subset['Zonal Wind'],
    multi_experiment_variables_subset['Vertical Wind']
]:

    meridional_mean_data[variable_data.name] = xr.concat(
        [
            variable_data.sel(experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])).mean(dim=['lat'])
            for experiment in coords.experiments.values
        ],
        dim=coords.experiments
    )

meridional_mean_data['Precipitation'] = multi_experiment_variables_subset['Precipitation']
for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_subset.items()):
# for index, (variable_name, variable_data) in enumerate(meridional_mean_data.items()):
# for index, (variable_name, variable_data) in enumerate(multi_experiment_budget_variables.items()):

    if not variable_name in variables_to_load:
        print(f"{variable_name} not in variables to load, skipping...")
        continue

    # print(f'{f"({index+1}/{len(meridional_mean_data)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")
    print(f'{f"({index+1}/{len(multi_experiment_variables_subset)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")
    # print(f'{f"({index+1}/{len(multi_experiment_budget_variables)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")

    # Remove the time mean of the variables to be regressed
    variable_data_anomalies = standardize_data(variable_data, unit_variance=False, dim='time')

    # Regress the variable anomalies onto the reference timeseries
    multi_experiment_variables_regressed[variable_name] = xr.dot(
        standardized_reference_timeseries,
        variable_data_anomalies,
        dim='time'
    ) / len(standardized_reference_timeseries.time) / multi_experiment_variables_subset['Precipitation'].sel(lat=slice(-15,15)).mean(dim=['time', 'lat', 'lon'])
    multi_experiment_variables_regressed[variable_name].attrs = variable_data_anomalies.attrs
    multi_experiment_variables_regressed[variable_name].name = variable_name

    # Calculate the statstical significance of the regression coefficients
    predicted_value = multi_experiment_variables_regressed[variable_name] * standardized_reference_timeseries
    residual_sum_of_squares = ((variable_data_anomalies - predicted_value) ** 2).sum(dim='time')
    residual_variance = residual_sum_of_squares / (N - 2)
    standard_error_slope = np.sqrt(residual_variance / (N - 1))
    t_statistic = multi_experiment_variables_regressed[variable_name] / standard_error_slope
    significant_regression_indices[variable_name] = np.abs(t_statistic) > t_critical

    print(rf"{'✔':>1}")

print(f"{'-'*config.SEP_WIDTH}")

# print("Re-scaling regressed timeseries")
# rescaled_multi_experiment_variables_regressed = {}
# rescaled_multi_experiment_variables_regressed = copy.deepcopy(multi_experiment_variables_regressed)
# print(f"{'-'*config.SEP_WIDTH}")

# rescaling_constant = {}
# rescaling_constant['Outgoing Longwave Radiation'] = -40
# rescaling_constant['Precipitation'] = 15
# rescaling_constant['Upper Level Moisture'] = 10
# rescaling_constant['Lower Level Moisture'] = 10

# for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_regressed.items()):
#     print(f'{f"({index+1}/{len(multi_experiment_variables_regressed)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")

#     for ((experiment_index, experiment), latitude, longitude) in zip(enumerate(coords.experiments.values), reference_latitude, reference_longitude):
#         # print(f'{f"{experiment}...":<{config.SEP_WIDTH-1}}', end="")
#         rescaling_factor = (
#             rescaling_constant[reference_variable]
#             / multi_experiment_variables_regressed[reference_variable].sel(experiment=experiment).sel(lat=latitude, lon=longitude, method='nearest')
#         )
#         rescaled_multi_experiment_variables_regressed[variable_name][experiment_index] = (
#             rescaling_factor.copy(deep=True)
#             * multi_experiment_variables_regressed[variable_name].sel(experiment=experiment)
#         )
#         rescaled_multi_experiment_variables_regressed[variable_name].attrs['rescaled'] = 'True'
#     print(rf"{'✔':>1}")
# print(f"{'-'*config.SEP_WIDTH}")

# if save_regressed_variables:
#     print(f"{'Saving regressed variables':^{config.SEP_WIDTH}}")
#     print(f"{'='*config.SEP_WIDTH}")

#     for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_regressed.items()):
#         print(f"{f'({index+1}/{len(multi_experiment_variables_regressed)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
#         multi_experiment_variables_regressed[variable_name].name = variable_name
#         significant_regression_indices[variable_name].name = variable_name

#         filename = (
#             f"{data_directory}/multi_experiment_variables_regressed/"
#             + f"multi_experiment_{variable_name.lower().replace(' ', '_')}_"
#             # + f"MJO-{rescaled_multi_experiment_variables_regressed[reference_variable].attrs['file_id']}.nc"
#             + f"MJO-{multi_experiment_variables_mjo_filtered[reference_variable].attrs['file_id']}.nc"
#         )
#         combined_data = xr.Dataset(
#             {
#                 f"{variable_name} - Coefficients": multi_experiment_variables_regressed[variable_name],
#                 f"{variable_name} - Significance": significant_regression_indices[variable_name]
#             }
#         )
#         if os.path.exists(filename):
#             # Prompt user for confirmation
#             # response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
#             response = 'y'
#             if response == 'y':
#                 os.remove(filename)  # Delete the existing file

#                 # combined_data["name"] = variable_data.name
#                 combined_data.to_netcdf(filename)  # Save the new file
#                 print(rf"{'✔ (overwritten)':>1}")
#             else:
#                 print(rf"{'✘ (skipped)':>1}")
#         else:
#             combined_data.to_netcdf(filename)  # Save the new file
#             print(rf"{'✔':>1}")
# else:
#     print(f"{'Not saving regressed variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
variables_to_regress = {
    'Moisture': multi_experiment_budget_variables['Moisture'],
    'Moisture Tendency': multi_experiment_budget_variables['Moisture Tendency'],
    'Precipitation': multi_experiment_budget_variables['Precipitation'],
    'Evaporation': multi_experiment_budget_variables['Evaporation'],
    'Residual': multi_experiment_budget_variables['Residual'],
    'Zonal Advection': multi_experiment_variables_subset['Zonal Moisture Advection'],
    'Meridional Advection': multi_experiment_variables_subset['Meridional Moisture Advection'],
    'Vertical Advection': multi_experiment_variables_subset['Vertical Moisture Advection'],
}

# Initialize the dictionary to hold the regressed values
multi_experiment_variables_regressed_moisture = {}
significant_regression_indices_moisture = {}

print(f"{'Regressions':^{config.SEP_WIDTH}}")
print(f"{f'Regression Variable: {reference_variable}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

for reference_timeseries in [multi_experiment_lower_level_moisture, multi_experiment_upper_level_moisture]:
    print(reference_timeseries.name)
    # Concatenate along the 'experiment' dimension
    standardized_reference_timeseries = standardize_data(reference_timeseries, unit_variance=True, dim='time')

    N = len(standardized_reference_timeseries.time)
    t_critical = t.ppf(0.95, df=N-2)

    significant_regression_indices = {}

    multi_experiment_variables_regressed_moisture[reference_timeseries.name] = {}
    significant_regression_indices_moisture[reference_timeseries.name] = {}

    for index, (variable_name, variable_data) in enumerate(variables_to_regress.items()):

        print(f'{f"({index+1}/{len(variables_to_regress)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")

        # Remove the time mean of the variables to be regressed
        variable_data_anomalies = standardize_data(variable_data, unit_variance=False, dim='time')

        # Regress the variable anomalies onto the reference timeseries
        multi_experiment_variables_regressed_moisture[reference_timeseries.name][variable_name] = xr.dot(
            standardized_reference_timeseries,
            variable_data_anomalies,
            dim='time'
        ) / len(standardized_reference_timeseries.time)
        multi_experiment_variables_regressed_moisture[reference_timeseries.name][variable_name].attrs = variable_data_anomalies.attrs
        multi_experiment_variables_regressed_moisture[reference_timeseries.name][variable_name].name = variable_name

        # Calculate the statstical significance of the regression coefficients
        predicted_value = multi_experiment_variables_regressed_moisture[reference_timeseries.name][variable_name] * standardized_reference_timeseries
        residual_sum_of_squares = ((variable_data_anomalies - predicted_value) ** 2).sum(dim='time')
        residual_variance = residual_sum_of_squares / (N - 2)
        standard_error_slope = np.sqrt(residual_variance / (N - 1))
        t_statistic = multi_experiment_variables_regressed_moisture[reference_timeseries.name][variable_name] / standard_error_slope
        significant_regression_indices_moisture[reference_timeseries.name][variable_name] = np.abs(t_statistic) > t_critical

        print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Correlate variables

In [ ]:
save_correlated_variables = False
# reference_variable = 'Precipitation'
data_type = 'correlated'
reference_variable = 'Upper Level Moisture'

# Initialize the dictionary to hold the correlated values
multi_experiment_variables_correlated = {}

# # Identify the location of the reference time series
reference_latitude = (
    multi_experiment_variables_mjo_filtered['Precipitation']
).var(dim='time').mean(dim='lon').idxmax(dim='lat')
reference_longitude = 180*xr.ones_like(reference_latitude)

reference_latitude = xr.zeros_like(reference_latitude)
reference_longitude = 180*xr.ones_like(reference_latitude)

# selected_data = [
#     ds.sel(
#         lon=reference_longitude.sel(experiment=experiment),
#         lat=reference_latitude.sel(experiment=experiment),
#         method="nearest"
#     )
#     for experiment, ds in zip(experiments_list, multi_experiment_variables_mjo_filtered[reference_variable])
#     # for experiment, ds in zip(experiments_list, mjo_filtered_moisture.sel(plev=slice(100,975)).integrate('plev'))
# ]

# # Concatenate along the 'experiment' dimension
# reference_timeseries = xr.concat(selected_data, dim="experiment")
# reference_longitude = 180
# reference_latitude = 0
reference_timeseries = multi_experiment_variables_mjo_filtered[reference_variable]
standardized_reference_timeseries = standardize_data(reference_timeseries, unit_variance=True, dim='time')

N = len(standardized_reference_timeseries.time)
t_critical = t.ppf(0.95, df=N-2)

significant_correlation_indices = {}

print(f"{'Correlations':^{config.SEP_WIDTH}}")
print(f"{f'Correlation Variable: {reference_variable}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

# for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_subset.items()):
for index, (variable_name, variable_data) in enumerate(multi_experiment_budget_variables.items()):

    # if not variable_name in variables_to_load:
    #     print(f"{variable_name} not in variables to load, skipping...")
    #     continue

    print(f'{f"({index+1}/{len(multi_experiment_budget_variables)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")

    # Remove the time mean of the variables to be correlated
    variable_data_anomalies = standardize_data(variable_data, unit_variance=True, dim='time')

    # Regress the variable anomalies onto the reference timeseries
    multi_experiment_variables_correlated[variable_name] = xr.dot(
        standardized_reference_timeseries,
        variable_data_anomalies,
        dim='time'
    ) / len(standardized_reference_timeseries.time)
    multi_experiment_variables_correlated[variable_name].attrs = variable_data_anomalies.attrs
    multi_experiment_variables_correlated[variable_name].name = variable_name

    # Calculate the statstical significance of the correlation coefficients
    predicted_value = multi_experiment_variables_correlated[variable_name] * standardized_reference_timeseries
    residual_sum_of_squares = ((variable_data_anomalies - predicted_value) ** 2).sum(dim='time')
    residual_variance = residual_sum_of_squares / (N - 2)
    standard_error_slope = np.sqrt(residual_variance / (N - 1))
    t_statistic = multi_experiment_variables_correlated[variable_name] / standard_error_slope
    significant_correlation_indices[variable_name] = np.abs(t_statistic) > t_critical

    print(rf"{'✔':>1}")

print(f"{'-'*config.SEP_WIDTH}")

if save_correlated_variables:
    print(f"{'Saving correlated variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_correlated.items()):
        print(f"{f'({index+1}/{len(multi_experiment_variables_correlated)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
        multi_experiment_variables_correlated[variable_name].name = variable_name
        significant_correlation_indices[variable_name].name = variable_name

        filename = (
            f"{data_directory}/multi_experiment_variables_correlated/"
            + f"multi_experiment_{variable_name.lower().replace(' ', '_')}_"
            + f"MJO-{multi_experiment_variables_mjo_filtered[reference_variable].attrs['file_id']}.nc"
        )
        combined_data = xr.Dataset(
            {
                f"{variable_name} - Coefficients": multi_experiment_variables_correlated[variable_name],
                f"{variable_name} - Significance": significant_correlation_indices[variable_name]
            }
        )
        if os.path.exists(filename):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            # response = 'y'
            if response == 'y':
                os.remove(filename)  # Delete the existing file

                # combined_data["name"] = variable_data.name
                combined_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            combined_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving correlated variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

# Plot regressed variables

## Vertical EOF Analysis

In [13]:
recalculate_EOFs = False

multi_experiment_EOF = {}
multi_experiment_explained_variance = {}

print("Computing EOFs")
print('=' * config.SEP_WIDTH)

for variable_name in ['Vertical Wind']:

    print(f"{f'{variable_name}...':<{config.SEP_WIDTH-1}}", end="")
    if variable_name in multi_experiment_EOF and not recalculate_EOFs:
        print(rf"{'✔':>1}")
        continue

    components = {}
    explained_variance_ratio = {}
    for experiment in coords.experiments.values:
        model = xe.single.EOF()
        model.fit(multi_experiment_variables_filtered[variable_name].sel(
            experiment=experiment,
            lat=slice(-15, 15),
            plev=slice(100, 950)
        ),
        dim=['time', 'lat', 'lon']
    )
        components[experiment] = model.components()
        explained_variance_ratio[experiment] = model.explained_variance_ratio()

    multi_experiment_EOF[variable_name] = xr.concat(
        [components[experiment] for experiment in coords.experiments.values],
        dim=coords.experiments,
    )
    multi_experiment_EOF[variable_name].name = 'EOFs'

    multi_experiment_explained_variance[variable_name] = xr.concat(
        [explained_variance_ratio[experiment] for experiment in coords.experiments.values],
        dim=coords.experiments,
    )
    multi_experiment_explained_variance[variable_name].name = 'Explained Variance'
    print(rf"{'✔':>1}")

print('=' * config.SEP_WIDTH)
print("Finished")

Computing EOFs
Vertical Wind...                       ✔
Finished


## Data parameters

In [21]:
# Specify bounds
longitude_bounds = {}
longitude_bounds['-4K'] = (0, 360)
longitude_bounds['0K'] = (0, 360)
longitude_bounds['4K'] = (0, 360)

wind_level = 'Lower Level'

if wind_level == 'Lower Level':
    center_level = [800., 800., 750.]

elif wind_level == 'Upper Level':
    center_level = [250, 200, 150]

plev_to_plot = {}
plev_to_plot['-4K'] = slice(center_level[0]-50, center_level[0]+50)
plev_to_plot['0K'] = slice(center_level[1]-50, center_level[1]+50)
plev_to_plot['4K'] = slice(center_level[2]-50, center_level[2]+50)

# plev_to_plot['-4K'] = slice(825, 950)
# plev_to_plot['0K'] = slice(775, 950)
# plev_to_plot['4K'] = slice(750, 950)

# plev_to_plot['-4K'] = slice(600-25, 600+25)
# plev_to_plot['0K'] = slice(525-25, 525+25)
# plev_to_plot['4K'] = slice(450-25, 450+25)

# Plot plev-specific variables at the level of the maximum moisture anomaly
if 'Moisture' in multi_experiment_variables_regressed:
    level_of_max_moisture = {}
    for experiment in coords.experiments.values:
        data_selection = {
            'experiment': experiment,
            # 'plev': slice(300,700),
            'lat': slice(*coords.latitude_bounds.sel(experiment=experiment)),
            # 'lon': slice(*longitude_bounds[experiment])
        }
        indices = np.unravel_index(
            multi_experiment_variables_regressed['Moisture'].sel(**data_selection).mean(dim='lat').argmax(),
            multi_experiment_variables_regressed['Moisture'].sel(**data_selection).mean(dim='lat').shape
        )
        level_of_max_moisture[experiment] = (
            multi_experiment_variables_regressed['Moisture'].sel(
                **data_selection
            ).mean(dim='lat')[indices].plev.values
        )
    print(f"Level of max. moisture: {level_of_max_moisture}")

print("Finished")

Level of max. moisture: {'-4K': array(825), '0K': array(525), '4K': array(450)}
Finished


In [35]:
max_v = {}
max_q = {}
for experiment in coords.experiments.values:
    max_v[experiment] = multi_experiment_variables_regressed['Meridional Wind'].sel(experiment=experiment, plev=plev_to_plot[experiment]).mean(dim='plev').sel(lat=15, lon=240, method='nearest').values
    max_q[experiment] = multi_experiment_variables_regressed['Moisture'].sel(experiment=experiment, plev=plev_to_plot[experiment]).mean(dim='plev').sel(lat=15, lon=240, method='nearest').values

In [37]:
print(max_v)
print(max_q)

{'-4K': array(0.08121848), '0K': array(0.1347363), '4K': array(0.25040021)}
{'-4K': array(-2.61897567e-05), '0K': array(-7.47636824e-05), '4K': array(-1.64914469e-05)}


In [39]:
print(max_v['4K']/max_v['0K'])
print(max_q['4K']/max_q['0K'])

1.8584464684708302
0.22058098790984498


## Latitude-Longitude

### Single reference variable

In [ ]:
multi_experiment_significant_indices = significant_regression_indices

In [ ]:
lat_skip_list = [
    -31.263158,
    -27.473684,
    -23.684211,
    -19.894737,
    -16.105263,
    -12.315789,
    -8.526316,
    -4.736842,
    -0.947368,
    0.947368,
    4.736842,
    8.526316,
    12.315789,
    16.105263,
    19.894737,
    23.684211,
    27.473684,
    31.263158
]

# multi_experiment_variables_regressed['Projected Vertical Wind'] = xr.dot(
#     multi_experiment_EOF['Vertical Wind'].sel(mode=1, drop=True),
#     multi_experiment_variables_regressed['Vertical Wind'],
#     dim='plev'
# )
# multi_experiment_variables_regressed['Projected Vertical Wind'].name = 'Projected Vertical Wind'
# multi_experiment_variables_regressed['Projected Vertical Wind'].attrs['units'] = ''

# Set plotting parameters
savefig = False
statistically_significant = False
plot_bounds = False
contour_variable = 'Column Water Vapor'
# contour_variable = 'Moisture'
# contour_variable = 'Precipitation'

contourf_variable = 'Precipitation'
# contourf_variable = 'Moisture'
# contourf_variable = 'Column Water Vapor'
# contourf_variable = 'Projected Vertical Wind'
# contour_variable = 'Meridional Moisture Advection'

# Scale variables to be plotted
contourf_scale = 1
contour_scale = 1
if data_type == 'regressed':
    if 'Moisture Advection' in contourf_variable:
        contourf_scale = 1000*config.SECONDS_PER_DAY
        # contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contourf_variable:
        contourf_scale = 1000

    if 'Moisture Advection' in contour_variable:
        contour_scale = 1000*config.SECONDS_PER_DAY
        # contour_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contour_variable:
        contour_scale = 1000

contourf_data = contourf_scale*multi_experiment_variables_regressed[contourf_variable].where(
        multi_experiment_significant_indices[contourf_variable]
        if statistically_significant else True
)

contour_data = contour_scale*multi_experiment_variables_regressed[contour_variable].where(
        multi_experiment_significant_indices[contour_variable]
        if statistically_significant else True
)

#### Wind Variables
zonal_wind_data = multi_experiment_variables_regressed['Zonal Wind'].where(
    multi_experiment_significant_indices['Zonal Wind']
    if statistically_significant else True
)
meridional_wind_data = multi_experiment_variables_regressed['Meridional Wind'].where(
    multi_experiment_significant_indices['Meridional Wind']
    if statistically_significant else True
)

# plev_to_plot = level_of_max_moisture

# 325., 450., 400.
# plev_to_plot['-4K'] = 600
# plev_to_plot['0K'] = 650
# plev_to_plot['4K'] = 750

# # Calculate levels automatically
# grand_max = contourf_data.sel(
#     **({'plev': [plev for plev in plev_to_plot.values()]} if 'plev' in contourf_data.coords else {})
# ).max()
# grand_min = contourf_data.sel(
#     **({'plev': [plev for plev in plev_to_plot.values()]} if 'plev' in contourf_data.coords else {})
# ).min()

# data_order = min(
#     np.floor(np.log10(np.abs(grand_min))),
#     np.floor(np.log10(np.abs(grand_max)))
# )
# step = 1
# increment = 0
# levels = np.arange(
#     round_out(grand_min, data_order+increment, step=step),
#     round_out(grand_max, data_order+increment, step=step)+(10**data_order),
#     (10**data_order)
# )
# index = 1
# while len(levels) < 11:
#     levels = np.arange(
#     round_out(grand_min, data_order+increment, step=step),
#     round_out(grand_max, data_order+increment, step=step)+((10**data_order)/(2**index)),
#     (10**data_order)/(2**index)
#     )
#     index += 1
# levels -= np.min(np.abs(levels))
# # levels = np.arange(-2.5, 2+0.25, 0.25)

# levels = {}
# levels['-4K'] = np.linspace(-0.35, 0.8, 17)
# levels['0K'] = np.linspace(-1, 1.75, 17)
# levels['4K'] = np.linspace(-3, 6, 17)


# Create figure
fontsize = 12
plt.rcParams.update({'font.size':fontsize})
cmap = modified_colormap('BrBG', 'white', 0.05, 0.05)
cmap.set_bad('white')
fig = plt.figure(figsize=(8, 12))
gs = GridSpec(3, 2, width_ratios=[30,1], height_ratios=[1,1,1], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.35, wspace=0.2)

# Create axes
axes = []
axes.append(fig.add_subplot(gs[0,0]))
axes.append(fig.add_subplot(gs[1,0]))
axes.append(fig.add_subplot(gs[2,0]))
cbar_ax = []
cbar_ax.append(fig.add_subplot(gs[0,1]))
cbar_ax.append(fig.add_subplot(gs[1,1]))
cbar_ax.append(fig.add_subplot(gs[2,1]))

# Plot data
for index, (ax, experiment) in enumerate(zip(axes, coords.experiments.values)):

    ax.set_title(
        (
            f"{string.ascii_letters[index+3]}) "
            f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}"
            # f" {plev_to_plot[experiment]} hPa"
            f": {plev_to_plot[experiment].start:.0f}-{plev_to_plot[experiment].stop:.0f} hPa"
        ),
        loc='left',
        fontsize=fontsize+2
    )
    ax.set_facecolor('white')

    # Add cyclic point to contourf data
    cdata = add_cyclic_xarray(
            contourf_data.sel(
                experiment=experiment,
                **(
                    {'plev': plev_to_plot[experiment]}
                    if 'plev' in multi_experiment_variables_regressed[contourf_variable].coords
                    else {})
            ),#.mean(dim='plev'),,
            'lon',
        )

    # Plot contourf data
    im = ax.contourf(
        cdata.lon,
        cdata.lat,
        cdata,
        cmap=cmap,
        # levels=levels,
        levels=15,
        norm=mcolors.CenteredNorm(vcenter=0),
        # extend='both'
    )
    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cbar_ax[index],
        label=contourf_data.attrs['units'],
        orientation="vertical",
    )
    cbar.ax.tick_params(labelsize=fontsize)

    # Plot contour data
    if contour_variable != 'None':

        # Add cyclic point to contour data
        cq = add_cyclic_xarray(
            contour_data.sel(
                experiment=experiment,
                **({'plev': plev_to_plot[experiment]} if 'plev' in multi_experiment_variables_regressed[contour_variable].coords else {})
            ),#.mean(dim='plev'),
            'lon'
        )

        cs = ax.contour(
            cq.lon,
            cq.lat,
            cq,
            colors='black',
            levels=(im.levels[im.levels != 0] if contour_variable == contourf_variable else 9),
            # linewidths=0.5,
            linewidths=1,
            alpha=0.75
        )
        if contour_variable != contourf_variable:
            ax.clabel(cs, inline=True, fontsize=6)


    # Add cyclic point to wind data
    cu = add_cyclic_xarray(
        zonal_wind_data.sel(
        # projected_zonal_wind.sel(
            experiment=experiment,
            # mode=2
            plev=plev_to_plot[experiment]
        ).mean(dim='plev'),
        # ),
        # ).integrate('plev')*100,
        'lon'
    )

    cv = add_cyclic_xarray(
        meridional_wind_data.sel(
        # projected_meridional_wind.sel(
            experiment=experiment,
            # mode=2
            plev=plev_to_plot[experiment]
        ).mean(dim='plev'),
        # ),
        # ).integrate('plev')*100,
        'lon'
    )

    # Plot wind data
    lon_skip = 4
    lon_skip_list = cu.lon[::lon_skip]
    scale = 0.5
    # scale = 1.5
    # scale=1000
    quiv = ax.quiver(
        cu.lon.sel(lon=lon_skip_list, method='nearest'),
        cv.lat.sel(lat=lat_skip_list, method='nearest'),
        cu.sel(lat=lat_skip_list, lon=lon_skip_list, method='nearest'),
        cv.sel(lat=lat_skip_list, lon=lon_skip_list, method='nearest'),
        scale_units='xy',
        angles='xy',
        # scale=scale,
        scale=None,
        width=0.0015,
        headwidth=6,
        headlength=10,
        minlength=2,
        zorder=10,
        color='black'
    )
    ax.figure.canvas.draw()
    quiv_multiplier = 17
    # scale_label = [1, 2, 5]
    # scale_label = quiv.scale
        # ax.quiverkey(quiv, X=0.9, Y=1.05, U=scale_label, label=f"{scale_label} m/s", labelpos='E')
    # ax.quiverkey(quiv, X=0.9, Y=1.05, U=scale_label[index], label=f"{scale_label[index]} m/s", labelpos='E')
    ax.quiverkey(
        quiv,
        X=0.87, Y=1.05,
        U=quiv_multiplier*quiv.scale,
        label=f"{quiv_multiplier*quiv.scale:0.1f} m/s",
        labelpos='E',
        coordinates='axes'
    )

    # Set axis parameters
    ax.set_aspect("auto")
    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks, labels=tick_labeller(x_ticks, "lon", precision=0))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(-30, 30)
    y_ticks = np.arange(-30, 45, 15)
    ax.set_yticks(y_ticks, labels=tick_labeller(y_ticks, "lat", precision=0))
    ax.set_ylabel("Latitude")

    ax.grid(True, **{"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"})

    if plot_bounds:
        ax.add_patch(
            patches.Rectangle(
                (longitude_bounds[experiment][0], coords.latitude_bounds[experiment][0]),
                longitude_bounds[experiment][1]-longitude_bounds[experiment][0],
                coords.latitude_bounds[experiment][1]-coords.latitude_bounds[experiment][0],
                linewidth=1,
                edgecolor='k',
                facecolor='lightgray',
                alpha=0.25
            )
        )


# if plev_to_plot != 'None':
#     # fig.suptitle(f"{plev_to_plot}-hPa {contourf_data.name} (colors) \n& {plev_to_plot}-hPa {contour_data.name} (contours) \n regressed onto {reference_variable}", y=1.025)
#     fig.suptitle(
#     (f"{plev_to_plot}-hPa {contourf_data.name} (colors), {plev_to_plot}-hPa {contour_data.name} (contours)" + (f"\n & Winds (vectors) " if data_type == 'regressed' else "\n ")
#     + f"{('regressed onto' if data_type == 'regressed' else 'correlated with')} Precipitation"),
#     x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#     ha='center',
#     y=1.025,
#     fontsize=16
#     )
# else:
#     # fig.suptitle(f"{contourf_data.name} (colors) \n& {contour_data.name} (contours) \n regressed onto {reference_variable}", y=1.025)
#     fig.suptitle(
#         (f"{contourf_data.name} (colors), {contour_data.name} (contours)" + (f"\n & Winds (vectors) " if data_type == 'regressed' else "\n ")
#         + f"{('regressed onto' if data_type == 'regressed' else 'correlated with')} Precipitation"),
#         x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#         ha='center',
#         y=1.025,
#         fontsize=16
#     )
# fig.suptitle(
#         (f"{contourf_data.name} (colors), {contour_data.name} (contours)" + (f"\n & Winds (vectors) " if data_type == 'regressed' else "\n ")
#         + f"{('regressed onto' if data_type == 'regressed' else 'correlated with')} Precipitation"),
#         x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#         ha='center',
#         y=1.025,
#         fontsize=16
#     )

# #### Set suptitle
# fig.suptitle(
#     (f"{contourf_data.name} (colors), {contour_data.name} (contours)" + (f"\n & Winds (vectors) " if data_type == 'regressed' else "\n ")
#     + f"{('regressed onto' if data_type == 'regressed' else 'correlated with')} Precipitation"),
#     x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#     ha='center',
#     y=1.025,
#     fontsize=16
# )

# fig.suptitle(
#     f'Precipitation & {wind_level} Winds',
#     x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#     ha='center',
#     y=1.0005,
#     fontsize=fontsize+2
# )

# Save figure
# output_filename = (
#     f"/mjo_{data_type}_"
#     f"{('rescaled_' if rescaled and data_type == 'regressed' else '')}"
#     f"{contourf_variable.lower().replace(" ", "_")}"
#     f"_{reference_variable}_latitude_longitude"
# )
# output_filename = 'Horizontal Structure of MJO-Regressed Precipitation, Column Water Vapor and Winds'
output_filename = f'Horizontal Structure of MJO-Regressed Precipitation and {wind_level} Winds'
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/")
print(f"Output filename: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

### Multiple reference variables

In [ ]:
savefig = False
statistically_significant = True

longitude_bounds = {}
longitude_bounds['-4K'] = (30, 260)
longitude_bounds['0K'] = (30, 300)
longitude_bounds['4K'] = (60, 300)

latitude_bounds = {}
latitude_bounds['-4K'] = (-5, 5)
latitude_bounds['0K'] = (-10, 10)
latitude_bounds['4K'] = (-15, 15)

fontsize = 16
plt.style.use('default')
plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(24, 12))
gs_main = fig.add_gridspec(1, 2, width_ratios = [45, 1], wspace=0.075)
gs_main.update(left=0.05, right=0.95, bottom=0.05, top=0.95)
gs_axes = gs_main[0].subgridspec(2, 3)
gs_cbar = gs_main[1].subgridspec(1, 1)

axes = [[],[]]
axes[0].append(fig.add_subplot(gs_axes[0,0]))
axes[0].append(fig.add_subplot(gs_axes[0,1]))
axes[0].append(fig.add_subplot(gs_axes[0,2]))

axes[1].append(fig.add_subplot(gs_axes[1,0]))
axes[1].append(fig.add_subplot(gs_axes[1,1]))
axes[1].append(fig.add_subplot(gs_axes[1,2]))

# cbar_ax = fig.add_subplot(gs[:, 1])
# cbar_axes = []
cbar_axes = fig.add_subplot(gs_cbar[0])
# cbar_axes.append(fig.add_subplot(gs_cbar[1]))

data_type = 'regressed'
contour_variable = 'Column Water Vapor'
# contourf_variable = 'Evaporation'
contourf_variable = 'Upper Level Moisture'
plev_to_plot = 'None'
# plev_to_plot = 850

if data_type == 'regressed':
    #### Contourf variable
    if 'Moisture Advection' in contourf_variable:
        ### Advection Variables
        contourf_data = 1000*config.SECONDS_PER_DAY*multi_experiment_variables_regressed[contourf_variable].where(
             (multi_experiment_significant_indices[contourf_variable] if statistically_significant else True)
        ).sel(plev=plev_to_plot)
        contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"

    else:
        contourf_data = multi_experiment_variables_regressed[contourf_variable].where(
            (multi_experiment_significant_indices[contourf_variable] if statistically_significant else True)
        ).sel(
            **({'plev': plev_to_plot} if 'plev' in multi_experiment_variables_regressed[contourf_variable].coords else {})
        )

    #### Contour Variable
    if contour_variable != 'None':
        contour_data = multi_experiment_variables_regressed[contour_variable].where(
            (multi_experiment_significant_indices[contour_variable] if statistically_significant else True)
        ).sel(
            **({'plev': plev_to_plot} if 'plev' in multi_experiment_variables_regressed[contour_variable].coords else {})
        )

    #### Wind Variables
    zonal_wind_data = multi_experiment_variables_regressed['Zonal Wind'].where(
        (multi_experiment_significant_indices['Zonal Wind'] if statistically_significant else True)
    ).sel(
        **({'plev': plev_to_plot} if plev_to_plot != 'None' else {'plev': 850})
    )
    meridional_wind_data = multi_experiment_variables_regressed['Meridional Wind'].where(
        (multi_experiment_significant_indices['Meridional Wind'] if statistically_significant else True)
    ).sel(
        **({'plev': plev_to_plot} if plev_to_plot != 'None' else {'plev': 850})
    )

elif data_type == 'correlated':
    #### Contourf variable
    contourf_data = multi_experiment_variables_correlated[contourf_variable].where(
            (multi_experiment_significant_indices[contourf_variable] if statistically_significant else True)
        ).sel(
            **({'plev': plev_to_plot} if 'plev' in multi_experiment_variables_correlated[contourf_variable].coords else {})
        )

    #### Contour Variable
    if contour_variable != 'None':
        contour_data = multi_experiment_variables_correlated[contour_variable].where(
            (multi_experiment_significant_indices[contour_variable] if statistically_significant else True)
        ).sel(
            **({'plev': plev_to_plot} if 'plev' in multi_experiment_variables_correlated[contour_variable].coords else {})
        )

    #### Wind Variables
    zonal_wind_data = multi_experiment_variables_correlated['Zonal Wind'].where(
        (multi_experiment_significant_indices['Zonal Wind'] if statistically_significant else True)
    ).sel(
        **({'plev': plev_to_plot} if plev_to_plot != 'None' else {'plev': 850})
    )
    meridional_wind_data = multi_experiment_variables_correlated['Meridional Wind'].where(
        (multi_experiment_significant_indices['Meridional Wind'] if statistically_significant else True)
    ).sel(
        **({'plev': plev_to_plot} if plev_to_plot != 'None' else {'plev': 850})
    )

if plev_to_plot != 'None':
    fig.suptitle(
        f"{plev_to_plot}-hPa {contourf_data.name} (colors), "
        + f"{(f'{plev_to_plot}-hPa ' if 'plev' in contour_data.dims else '')}{contour_data.name} (contours) "
        + (f" & {(f'{plev_to_plot}-hPa ' if 'plev' in contour_data.dims else '')}Winds (vectors) \n " if data_type == 'regressed' else "\n")
        + f"\n {('regressed onto' if data_type == 'regressed' else 'correlated with')} Upper (top) & Lower (bottom) level Moisture",
        y=1.05,
        fontsize=20
    )
else:
    fig.suptitle(
        f"{contourf_data.name} (colors), {contour_data.name} (contours)"
        + (f" & 850-hPa Winds (vectors)" if data_type == 'regressed' else "")
        + f"\n {('regressed onto' if data_type == 'regressed' else 'correlated with')} Upper (top) & Lower (bottom) level Moisture",
        y=1.05,
        fontsize=20
    )

letter = [['a', 'b', 'c'], ['d', 'e', 'f']]

for level_index, level in enumerate(multi_experiment_variables_regressed['Moisture'].level):
    for experiment_index, experiment in enumerate(coords.experiments.values):

        axes[level_index][experiment_index].set_title(
            f"{letter[level_index][experiment_index]}) "
            + f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}",
            loc='left'
        )
        axes[level_index][experiment_index].set_facecolor('white')

        cdata, clon = cutil.add_cyclic_point(
            contourf_data.sel(experiment=experiment, level=level),
            coord=contourf_data.lon,
        )

        grand_max = contourf_data.max()
        grand_min = contourf_data.min()
        data_order = min(
            np.floor(np.log10(np.abs(grand_min))),
            np.floor(np.log10(np.abs(grand_max)))
        )

        if data_type == 'regressed':
            levels = np.arange(
                round_out(grand_min, data_order+1),
                round_out(grand_max, data_order+1)+10**data_order,
                10**data_order
            )
            index = 1
            while len(levels) < 11:
                levels = np.arange(
                round_out(grand_min, data_order+1),
                round_out(grand_max, data_order+1)+(10**data_order/2**index),
                10**data_order/2**index
                )
                index += 1
            levels -= np.min(np.abs(levels))
        else:
            magnitude_max = max(np.abs(grand_min), np.abs(grand_max))
            step=1/4
            increment = 0
            levels = np.arange(
                -round_out(magnitude_max, -data_order-increment),
                round_out(magnitude_max, -data_order-increment)+10**data_order,
                10**data_order
            )
            index = 1
            while len(levels) < 15:
                levels = np.arange(
                    -round_out(magnitude_max, -data_order-increment, step=step),
                    round_out(magnitude_max, -data_order-increment, step=step)+10**data_order/2**index,
                    10**data_order/2**index
                )
                index += 1

            levels -= levels[np.argmin(np.abs(levels))]

        data_captured = (grand_max.values < np.max(levels))*(grand_min.values > np.min(levels))
        print(f"({level.values}, {experiment}) Colorbar captures all data? {data_captured}")
        cmap = modified_colormap('BrBG', 'white', 0.05, 0.05)
        cmap.set_bad('white')
        # Plot data
        im = axes[level_index][experiment_index].contourf(
            clon,
            contourf_data.lat,
            cdata,
            cmap=cmap,
            # levels=levels,
            levels = levels,
            norm=mcolors.CenteredNorm(vcenter=0, halfrange=max(np.abs(grand_min.values), np.abs(grand_max.values))),
            # extend='both'
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            # cax=cbar_axes[index],
            cax=cbar_axes,
            label=(contourf_data.attrs['units'] if data_type == 'regressed' else ''),
            orientation="vertical",
        )
        # cbar.set_ticks()
        cbar.ax.tick_params(labelsize=fontsize)

        if contour_variable != 'None':
            cq, clon = cutil.add_cyclic_point(
                contour_data.sel(experiment=experiment, level=level),
                coord=contour_data.lon,
            )

            cs = axes[level_index][experiment_index].contour(
                clon,
                contour_data.lat,
                cq,
                colors='black',
                levels=11,
                linewidths=0.5,
                alpha=0.75
            )
            ax.clabel(cs, inline=True, fontsize=10)

        if data_type == 'regressed':
            # Add cyclic point
            cu, clon = cutil.add_cyclic_point(
                zonal_wind_data.sel(experiment=experiment, level=level),
                coord=zonal_wind_data.lon,
            )

            cv, clon = cutil.add_cyclic_point(
                meridional_wind_data.sel(experiment=experiment, level=level),
                coord=meridional_wind_data.lon,
            )

            lon_skip = 4
            lat_skip = 2
            scale = 1
            quiv = axes[level_index][experiment_index].quiver(
                clon[::lon_skip],
                zonal_wind_data.lat[::lat_skip],
                cu[::lat_skip, ::lon_skip],
                cv[::lat_skip, ::lon_skip],
                # width=0.00125,
                scale_units='xy',
                angles='xy',
                scale=scale,
                width=0.0015,
                headwidth=6,
                headlength=10,
                minlength=2,
                zorder=10,
                color='black'
            )
            scale_label = 10
            axes[level_index][experiment_index].quiverkey(quiv, X=0.9, Y=1.05, U=scale_label, label=f"{scale_label} m/s", labelpos='E')

        # if reference_point is not None:
        #     ax.plot(reference_point[0], reference_point[1].sel(experiment=experiment), marker='x', ms=14, color='k')

        for ax in axes[level_index]:
            # Axis parameters
            ax.set_aspect("auto")

            # if experiment == '-4K':
            #     rect = patches.Rectangle((0, -5), 300, 10, linewidth=1, edgecolor='k')

            ax.set_xlim(0, 360)
            x_ticks = np.arange(0, 360 + 60, 60)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(tick_labeller(x_ticks, "lon", precision=0))
            ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
            ax.set_xlabel("Longitude")

            ax.set_ylim(-30, 30)
            y_ticks = np.arange(-30, 45, 15)
            ax.set_yticks(y_ticks)
            ax.set_yticklabels(tick_labeller(y_ticks, "lat", precision=0))
            ax.set_ylabel("Latitude")

            grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
            ax.grid(True, **grid_kwargs)

        # ax.add_patch(
        #     patches.Rectangle(
        #         (longitude_bounds[experiment][0], latitude_bounds[experiment][0]),
        #         longitude_bounds[experiment][1]-longitude_bounds[experiment][0],
        #         latitude_bounds[experiment][1]-latitude_bounds[experiment][0],
        #         linewidth=1,
        #         edgecolor='k',
        #         facecolor='lightgray',
        #         alpha=0.25
        #     )
        # )

# fig.suptitle(
#     fig_title,
#     x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#     ha='center',
#     y=1.025
# )

output_filename = (
    f"/mjo_{data_type}_"
    + f"{(f"{plev_to_plot}-hPa-" if plev_to_plot != 'None' else '')}{contourf_variable.lower().replace(" ", "_")}"
    + f"_{reference_variable}_longitude_latitude"
)
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}")
print(f"Output file: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/{output_filename}.png",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

## Longitude-Height

### Single reference variable

In [ ]:
savefig = False
statistically_significant = False

plt.style.use('default')
fontsize = 12
plt.rcParams.update({'font.size':fontsize})
# plev_to_plot = level_of_max_moisture
# plev_to_plot = {}
# plev_to_plot['-4K'] = 550
# plev_to_plot['0K'] = 475
# plev_to_plot['4K'] = 475

contourf_variable = 'Meridional Moisture Advection'
# contourf_variable = 'Moisture'
# contourf_variable = 'Vertical Wind'
# contourf_variable = 'Moisture Tendency'
contour_variable = 'Moisture'

contourf_scale = 1
contour_scale = 1
if data_type == 'regressed':
    if 'Moisture Advection' in contourf_variable:
        contourf_scale = 1000*config.SECONDS_PER_DAY
        # contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contourf_variable:
        contourf_scale = 1000

    if 'Moisture Advection' in contour_variable:
        contour_scale = 1000*config.SECONDS_PER_DAY
        # contour_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contour_variable:
        contour_scale = 1000

# contourf_data = contourf_scale*multi_experiment_variables_regressed[contourf_variable].transpose('experiment', 'lat', 'lon', 'plev')
contourf_data = (
    contourf_scale
    # * 1000*multi_experiment_variables_subset['Moisture'].mean(dim=['time', 'lat', 'lon'])
    * multi_experiment_variables_regressed[contourf_variable]
).transpose('experiment', 'lat', 'lon', 'plev')
contourf_data.name = contourf_variable
contourf_data.attrs['units'] = r"g kg$^{-1}$"
# contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
# contour_data = contour_scale*multi_experiment_variables_regressed[contour_variable].transpose('experiment', 'lat', 'lon', 'plev')
contour_data = (
    contour_scale
    # * 1000*multi_experiment_variables_subset['Moisture'].mean(dim=['time', 'lat', 'lon'])
    * multi_experiment_variables_regressed[contour_variable]
).transpose('experiment', 'lat', 'lon', 'plev')
contour_data.name = contour_variable
contour_data.attrs['units'] = r"g kg$^{-1}$"

# latitude_mask = xr.zeros_like((multi_experiment_variables_regressed['Vertical Wind']))
# # latitude_mask.loc[{'experiment':'-4K', 'lat':slice(-5,5)}] = 1
# # latitude_mask.loc[{'experiment':'0K', 'lat':slice(-10,10)}] = 1
# # latitude_mask.loc[{'experiment':'4K', 'lat':slice(-15,15)}] = 1
# latitude_mask.loc[{'lat':slice(-15,15)}] = 1

# latitude_bounds = {}
# latitude_bounds['-4K'] = (-15,15)
# latitude_bounds['0K'] = (-15,15)
# latitude_bounds['4K'] = (-15,15)

# Take a meridional mean over the specified domain
meridional_mean_data = {}
meridional_mean_data[contourf_variable] = contourf_data.where(coords.latitude_mask).mean(dim='lat')
meridional_mean_data[contour_variable] = contour_data.where(coords.latitude_mask).mean(dim='lat')
meridional_mean_data['Zonal Wind'] = multi_experiment_variables_regressed['Zonal Wind'].where(coords.latitude_mask).mean(dim='lat')
meridional_mean_data['Vertical Wind'] = multi_experiment_variables_regressed['Vertical Wind'].where(coords.latitude_mask).mean(dim='lat')

fontsize = 14
plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(8, 12))
gs = GridSpec(3, 2, width_ratios=[30,1], height_ratios=[1,1,1], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.35, wspace=0.2)

axes = []
axes.append(fig.add_subplot(gs[0,0]))
axes.append(fig.add_subplot(gs[1,0]))
axes.append(fig.add_subplot(gs[2,0]))
cbar_axes = []
cbar_axes.append(fig.add_subplot(gs[0,1]))
cbar_axes.append(fig.add_subplot(gs[1,1]))
cbar_axes.append(fig.add_subplot(gs[2,1]))

# Calculate nice levels automatically
grand_min, grand_max = meridional_mean_data[contourf_variable].min(), meridional_mean_data[contourf_variable].max()
data_order = min(
    np.floor(np.log10(np.abs(grand_min))),
    np.floor(np.log10(np.abs(grand_max)))
)
step = 1/2
increment = 4
levels = np.arange(
    round_out(grand_min, data_order+increment, step=step),
    round_out(grand_max, data_order+increment, step=step)+(10**data_order),
    (10**data_order)
)
index = 1
while len(levels) < 18:
    levels = np.arange(
    round_out(grand_min, data_order+increment, step=step),
    round_out(grand_max, data_order+increment, step=step)+((10**data_order)/(2**index)),
    (10**data_order)/(2**index)
    )
    index += 1
# levels -= np.min(np.abs(levels))
levels -= np.sign(levels[np.argmin(np.abs(levels))])*np.min(np.abs(levels))
# levels = np.arange(-2, 2+0.25,0.25)

# Plot data
for index, (ax, experiment) in enumerate(zip(axes, coords.experiments.values)):

    ax.set_title(
        f"{string.ascii_letters[index]}) {config.EXPERIMENT_DISPLAY_NAMES[experiment]}",
        # f" ({tick_labeller(latitude_bounds[experiment], 'lat', precision=0)[0]}"
        # f"-{tick_labeller(latitude_bounds[experiment], 'lat', precision=0)[1]})",
        loc='left',
        fontsize=fontsize+2
    )

    contourf_cdata, clon = cutil.add_cyclic_point(
        meridional_mean_data[contourf_variable].where(
            (significant_regression_indices[contourf_variable] if statistically_significant else True)
        ).sel(experiment=experiment).T,
        coord=meridional_mean_data[contourf_variable].lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        meridional_mean_data[contourf_variable].plev,
        contourf_cdata,
        cmap=modified_colormap('BrBG', 'white', 0.05, 0.05),
        # levels=levels,
        levels=17,
        norm=mcolors.CenteredNorm(
            vcenter=0,
            # halfrange=max(np.abs(grand_min.values),
            #               np.abs(grand_max.values))
        ),
        # extend='both'
    )
    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cbar_axes[index],
        # cax=cbar_ax,
        label=meridional_mean_data[contourf_variable].attrs['units'],
        orientation="vertical",
    )
    cbar.ax.tick_params(labelsize=fontsize)

    contour_cdata, clon = cutil.add_cyclic_point(
        meridional_mean_data[contour_variable].where(
            (significant_regression_indices[contour_variable] if statistically_significant else True)
        ).sel(experiment=experiment).T,
        coord=meridional_mean_data[contour_variable].lon,
    )

    cs = ax.contour(
        clon,
        meridional_mean_data[contour_variable].plev,
        contour_cdata,
        colors='black',
        levels=(im.levels[im.levels != 0] if contour_variable == contourf_variable else 11),
        # linewidths=0.5,
        linewidths=0.75,
        alpha=0.75
    )
    if contour_variable != contourf_variable:
        ax.clabel(cs, inline=True, fontsize=6)

    ax.contour(
        clon,
        meridional_mean_data[contour_variable].plev,
        contour_cdata,
        colors='black',
        levels=[0],
        # linewidths=0.5,
        linewidths=0.75,
        alpha=0.75
    )

    # cu, clon = cutil.add_cyclic_point(
    #     meridional_mean_data['Zonal Wind'].where(
    #         (significant_regression_indices['Zonal Wind'] if statistically_significant else True)
    #     ).sel(experiment=experiment).T,
    #     coord=meridional_mean_data['Zonal Wind'].lon,
    # )

    # comega, clon = cutil.add_cyclic_point(
    #     (3600/10)*meridional_mean_data['Vertical Wind'].where(
    #         (significant_regression_indices['Vertical Wind'] if statistically_significant else True)
    #     ).sel(experiment=experiment).T,
    #     coord=meridional_mean_data['Vertical Wind'].lon,
    # )

    # lon_spacing, height_spacing = 6, 3
    # scale = 0.1
    # quiv = ax.quiver(
    #     clon[::lon_spacing],
    #     meridional_mean_data['Vertical Wind'].plev[::height_spacing],
    #     cu[::height_spacing, ::lon_spacing],
    #     comega[::height_spacing, ::lon_spacing],
    #     scale_units='xy',
    #     angles='xy',
    #     # scale=scale,
    #     scale=None,
    #     width=0.0015,
    #     headwidth=6,
    #     headlength=10,
    #     minlength=2,
    #     zorder=10,
    #     color='black',
    #     alpha=0.7
    # )
    # # scale_label = 1
    # quiv_multiplier = 25
    # ax.figure.canvas.draw()
    # ax.quiverkey(
    #     quiv,
    #     X=0.87, Y=1.05,
    #     U=quiv_multiplier*quiv.scale,
    #     label=f"{quiv_multiplier*quiv.scale:0.1f} m/s",
    #     labelpos='E',
    #     coordinates='axes'
    # )

    # Axis parameters
    ax.set_aspect("auto")

    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks, labels = tick_labeller(x_ticks, "lon", precision=0))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(100, 950)
    ax.set_yticks(np.arange(100, 1000, 100))
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    # [ax.axvline(x=tick, color='#bcbcbc', alpha=0.9, ls=':') for tick in x_ticks]
    # ax.grid(axis='x', **grid_kwargs)
    # ax.axhline(y=plev_to_plot[experiment], color='black', alpha=0.65, ls='--', lw=1.25)
    # ax.add_patch(
    #     patches.Rectangle(
    #         (0, plev_to_plot[experiment].start),
    #         360,
    #         plev_to_plot[experiment].stop-plev_to_plot[experiment].start,
    #         linewidth=0.5,
    #         edgecolor='k',
    #         facecolor='lightgray',
    #         linestyle='--',
    #         alpha=0.5
    #     )
    # )

#### Set suptitle
# fig.suptitle(
#     (f"{contourf_data.name} (colors), {contour_data.name} (contours)" + (f"\n & Winds (vectors) " if data_type == 'regressed' else "\n ")
#     + f"{('regressed onto' if data_type == 'regressed' else 'correlated with')} Precipitation"),
#     x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
#     ha='center',
#     y=1.025,
#     fontsize=16
# )

fig.suptitle(
    (
        f'{contourf_variable} (Colors)\n & Moisture (Contours)'
        if contourf_variable != contour_variable else
        f"{contour_variable}"
    ),
    x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
    ha='center',
    y=1.0005,
    # y=1.033333333,
    fontsize=fontsize+2
)

# output_filename = (
#     f"/mjo_{data_type}_"
#     f"{('rescaled_' if rescaled and data_type == 'regressed' else '')}"
#     f"{contourf_variable.lower().replace(" ", "_")}"
#     f"_{reference_variable}_longitude_height"
# )
# output_filename = "Longitude-Height Plot of Meridional Moisture Advection, Moisture & Winds"
output_filename = "Longitude-Height Plot of Moisture & Winds - No Patches"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/")
print(f"Output file: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

In [ ]:
savefig = False
statistically_significant = True
fontsize = 14
plt.rcParams.update({'font.size':fontsize})
cmap = modified_colormap('BrBG', 'white', 0.05, 0.05)
cmap.set_bad('white')

longitude_bounds = {}
longitude_bounds['-4K'] = (30, 260)
longitude_bounds['0K'] = (30, 300)
longitude_bounds['4K'] = (60, 300)

# plev_to_plot = {}
# plev_to_plot['-4K'] = 850
# plev_to_plot['0K'] = 775
# plev_to_plot['4K'] = 750
# plev_to_plot['-4K'] = 600
# plev_to_plot['0K'] = 650
# plev_to_plot['4K'] = 750

contourf_variable = 'Precipitation'
# contourf_variable = 'Moisture'
# contourf_variable = 'Meridional Moisture Advection'
# contour_variable = 'Vertical Advection'
# contour_variable = 'Column Water Vapor'
contour_variable = 'Precipitation'

contourf_scale = 1
contour_scale = 1
if data_type == 'regressed':
    if 'Moisture Advection' in contourf_variable:
        contourf_scale = 1000*config.SECONDS_PER_DAY
        # contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contourf_variable:
        contourf_scale = 1000

    if 'Moisture Advection' in contour_variable:
        contour_scale = 1000*config.SECONDS_PER_DAY
        # contour_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contour_variable:
        contour_scale = 1000

contourf_data = contourf_scale*multi_experiment_variables_regressed[contourf_variable].where(
     (multi_experiment_significant_indices[contourf_variable] if statistically_significant else True)
)

contour_data = contour_scale*multi_experiment_variables_regressed[contour_variable].where(
    (multi_experiment_significant_indices[contour_variable] if statistically_significant else True)
)

#### Wind Variables
zonal_wind_data = multi_experiment_variables_regressed['Zonal Wind'].where(
    (multi_experiment_significant_indices['Zonal Wind'] if statistically_significant else True)
)
meridional_wind_data = multi_experiment_variables_regressed['Meridional Wind'].where(
    (multi_experiment_significant_indices['Meridional Wind'] if statistically_significant else True)
)

letter = ['a', 'b', 'c']

fig = plt.figure(figsize=(16, 12))
gs_main = fig.add_gridspec(1, 2, width_ratios=[1,1], wspace=0.35)
gs_left = gs_main[0].subgridspec(3, 2, width_ratios=[20,1], wspace=0.2, hspace=0.4)
gs_right = gs_main[1].subgridspec(3, 2, width_ratios=[20,1], wspace=0.2, hspace=0.4)

# gs = GridSpec(3, 4, width_ratios=[30, 1, 30, 1], height_ratios=[1,1,1], figure=fig)
# gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.35, wspace=0.3)

axes = []
axes.append(fig.add_subplot(gs_left[0,0]))
axes.append(fig.add_subplot(gs_left[1,0]))
axes.append(fig.add_subplot(gs_left[2,0]))
axes.append(fig.add_subplot(gs_right[0,0]))
axes.append(fig.add_subplot(gs_right[1,0]))
axes.append(fig.add_subplot(gs_right[2,0]))
cbar_ax = []
cbar_ax.append(fig.add_subplot(gs_left[:, -1]))
cbar_ax.append(fig.add_subplot(gs_right[:, -1]))

for index, (ax, experiment) in enumerate(zip(axes[:3], coords.experiments.values)):

    ax.set_title(f"{letter[index]}) {config.EXPERIMENT_DISPLAY_NAMES[experiment]}: {plev_to_plot[experiment]} hPa", loc='left')
    ax.set_facecolor('white')

    grand_max = contourf_data.sel(
        **({'plev': [plev for plev in plev_to_plot.values()]} if 'plev' in contourf_data.coords else {})
    ).max()
    grand_min = contourf_data.sel(
        **({'plev': [plev for plev in plev_to_plot.values()]} if 'plev' in contourf_data.coords else {})
    ).min()

    data_order = min(
        np.floor(np.log10(np.abs(grand_min))),
        np.floor(np.log10(np.abs(grand_max)))
    )
    step = 1/2
    increment = 2
    levels = np.arange(
        round_out(grand_min, data_order+increment, step=step),
        round_out(grand_max, data_order+increment, step=step)+(10**data_order),
        (10**data_order)
    )
    index = 1
    while len(levels) < 15:
        levels = np.arange(
        round_out(grand_min, data_order+increment, step=step),
        round_out(grand_max, data_order+increment, step=step)+((10**data_order)/(2**index)),
        (10**data_order)/(2**index)
        )
        index += 1
    # levels -= np.min(np.abs(levels))
    levels -= np.sign(levels[np.argmin(np.abs(levels))])*np.min(np.abs(levels))

    cdata, clon = cutil.add_cyclic_point(
            contourf_data.sel(experiment=experiment).sel(
                **({'plev': plev_to_plot[experiment]} if 'plev' in multi_experiment_variables_regressed[contourf_variable].coords else {})
            ),
            coord=contourf_data.lon,
        )

    # Plot data
    im = ax.contourf(
        clon,
        contourf_data.lat,
        cdata,
        cmap=cmap,
        levels=levels,
        norm=mcolors.CenteredNorm(vcenter=0),
        extend='both'
    )
    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cbar_ax[0],
        label=contourf_data.attrs['units'],
        orientation="vertical",
    )
    cbar.ax.tick_params(labelsize=fontsize)

    if contour_variable != 'None':
        cq, clon = cutil.add_cyclic_point(
            contour_data.sel(experiment=experiment).sel(
                **({'plev': plev_to_plot[experiment]} if 'plev' in multi_experiment_variables_regressed[contour_variable].coords else {})
            ),
            coord=contour_data.lon,
        )

        cs = ax.contour(
            clon,
            contour_data.lat,
            cq,
            colors='black',
            levels=(im.levels[im.levels != 0] if contour_variable == contourf_variable else 11),
            linewidths=0.5,
            alpha=0.75
        )
        # ax.clabel(cs, inline=True, fontsize=6)

    # Add cyclic point
    cu, clon = cutil.add_cyclic_point(
        zonal_wind_data.sel(experiment=experiment, plev=plev_to_plot[experiment]),
        coord=zonal_wind_data.lon,
    )

    cv, clon = cutil.add_cyclic_point(
        meridional_wind_data.sel(experiment=experiment, plev=plev_to_plot[experiment]),
        coord=meridional_wind_data.lon,
    )

    lon_skip = 4
    lat_skip = 2
    # scale = 0.75
    scale = 0.1
    quiv = ax.quiver(
        clon[::lon_skip],
        zonal_wind_data.lat[::lat_skip],
        cu[::lat_skip, ::lon_skip],
        cv[::lat_skip, ::lon_skip],
        scale_units='xy',
        angles='xy',
        scale=scale,
        width=0.0015,
        headwidth=6,
        headlength=10,
        minlength=2,
        zorder=10,
        color='black'
    )
    scale_label = 1
    ax.quiverkey(quiv, X=0.9, Y=1.05, U=scale_label, label=f"{scale_label} m/s", labelpos='E')

    # Axis parameters
    ax.set_aspect("auto")

    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks, "lon", precision=0))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(-30, 30)
    y_ticks = np.arange(-30, 45, 15)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(tick_labeller(y_ticks, "lat", precision=0))
    ax.set_ylabel("Latitude")

    grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
    ax.grid(True, **grid_kwargs)

    # ax.add_patch(
    #     patches.Rectangle(
    #         (longitude_bounds[experiment][0], latitude_bounds[experiment][0]),
    #         longitude_bounds[experiment][1]-longitude_bounds[experiment][0],
    #         latitude_bounds[experiment][1]-latitude_bounds[experiment][0],
    #         linewidth=1,
    #         edgecolor='k',
    #         facecolor='lightgray',
    #         alpha=0.25
    #     )
    # )
    ax.add_patch(
            patches.Rectangle(
                (0, latitude_bounds[experiment][0]),
                360,
                latitude_bounds[experiment][1]-latitude_bounds[experiment][0],
                linewidth=1,
                edgecolor='k',
                facecolor='lightgray',
                alpha=0.15
            )
        )


letter = ['d', 'e', 'f']

contourf_variable = 'Moisture'
contour_variable = 'Moisture'

contourf_scale = 1
contour_scale = 1
if data_type == 'regressed':
    if 'Moisture Advection' in contourf_variable:
        contourf_scale = 1000*config.SECONDS_PER_DAY
        # contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contourf_variable:
        contourf_scale = 1000

    if 'Moisture Advection' in contour_variable:
        contour_scale = 1000*config.SECONDS_PER_DAY
        # contour_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contour_variable:
        contour_scale = 1000

contourf_data = contourf_scale*multi_experiment_variables_regressed[contourf_variable]
# contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
contour_data = contour_scale*multi_experiment_variables_regressed[contour_variable]
contour_data.attrs['units'] = r"g kg$^{-1}$"

meridional_mean_contourf_data_by_experiment = []
meridional_mean_contour_data_by_experiment = []
meridional_mean_zonal_wind_by_experiment = []
meridional_mean_vertical_wind_by_experiment = []

statistically_significant = False

# Take a meridional mean over the specified domain
if not statistically_significant:
    for experiment in coords.experiments.values:
        meridional_mean_contourf_data_by_experiment.append(
            contourf_data.sel(
                experiment=experiment
            ).sel(
                lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim=['lat'])
        )

        meridional_mean_contour_data_by_experiment.append(
            contour_data.sel(
                experiment=experiment
            ).sel(
                lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim=['lat'])
        )

        meridional_mean_zonal_wind_by_experiment.append(
            multi_experiment_variables_regressed['Zonal Wind'].sel(
                experiment=experiment
            ).sel(
                lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim=['lat'])
        )
        meridional_mean_vertical_wind_by_experiment.append(
            multi_experiment_variables_regressed['Vertical Wind'].sel(
                experiment=experiment
            ).sel(
                lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim=['lat'])
        )

meridional_mean_contourf_data = xr.concat(
    meridional_mean_contourf_data_by_experiment,
    dim=coords.experiments
)

meridional_mean_contour_data = xr.concat(
    meridional_mean_contour_data_by_experiment,
    dim=coords.experiments
)

meridional_mean_zonal_wind = xr.concat(
    meridional_mean_zonal_wind_by_experiment,
    dim=coords.experiments
)
meridional_mean_vertical_wind = xr.concat(
    meridional_mean_vertical_wind_by_experiment,
    dim=coords.experiments
)

# rescaled_meridional_mean_zonal_wind = meridional_mean_zonal_wind / meridional_mean_zonal_wind.std(dim=['plev', 'lon'])
# rescaled_meridional_mean_vertical_wind = meridional_mean_vertical_wind / meridional_mean_vertical_wind.std(dim=['plev', 'lon'])

rescaled_meridional_mean_zonal_wind = meridional_mean_zonal_wind
rescaled_meridional_mean_vertical_wind = (3600/10)*meridional_mean_vertical_wind

grand_max = meridional_mean_contourf_data.max()
grand_min = meridional_mean_contourf_data.min()
data_order = min(
    np.floor(np.log10(np.abs(grand_min))),
    np.floor(np.log10(np.abs(grand_max)))
)
step = 1
increment = 2
levels = np.arange(
    round_out(grand_min, data_order+increment, step=step),
    round_out(grand_max, data_order+increment, step=step)+(10**data_order),
    (10**data_order)
)
index = 1
while len(levels) < 18:
    levels = np.arange(
    round_out(grand_min, data_order+increment, step=step),
    round_out(grand_max, data_order+increment, step=step)+((10**data_order)/(2**index)),
    (10**data_order)/(2**index)
    )
    index += 1
# levels -= np.min(np.abs(levels))
levels -= np.sign(levels[np.argmin(np.abs(levels))])*np.min(np.abs(levels))

for index, (ax, experiment) in enumerate(zip(axes[3:], coords.experiments.values)):

    ax.set_title(
        f"{letter[index]}) {config.EXPERIMENT_DISPLAY_NAMES[experiment]}"
        + f" ({tick_labeller(coords.latitude_bounds[experiment], 'lat', precision=0)[0]}"
        + f"-{tick_labeller(coords.latitude_bounds[experiment], 'lat', precision=0)[1]})",
        loc='left'
    )

    contourf_cdata, clon = cutil.add_cyclic_point(
        meridional_mean_contourf_data.sel(experiment=experiment).T,
        coord=meridional_mean_contourf_data.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        meridional_mean_contourf_data.plev,
        contourf_cdata,
        cmap=modified_colormap('BrBG', 'white', 0.05, 0.05),
        levels=levels,
        norm=mcolors.CenteredNorm(
            vcenter=0,
            halfrange=max(np.abs(grand_min.values),
                          np.abs(grand_max.values))
        ),
        extend='both'
    )
    # Add colorbar
    cbar = fig.colorbar(
        im,
        # cax=cbar_axes[index],
        cax=cbar_ax[1],
        label=meridional_mean_contourf_data.attrs['units'],
        orientation="vertical",
    )
    cbar.ax.tick_params(labelsize=fontsize)

    contour_cdata, clon = cutil.add_cyclic_point(
        meridional_mean_contour_data.sel(experiment=experiment).T,
        coord=meridional_mean_contour_data.lon,
    )

    cs = ax.contour(
        clon,
        meridional_mean_contour_data.plev,
        contour_cdata,
        colors='black',
        levels=(im.levels[im.levels != 0] if contour_variable == contourf_variable else 11),
        linewidths=0.5,
        alpha=0.75
    )
    # ax.clabel(cs, inline=True, fontsize=10)

    cu, clon = cutil.add_cyclic_point(
        rescaled_meridional_mean_zonal_wind.sel(experiment=experiment).T,
        coord=meridional_mean_zonal_wind.lon,
    )

    comega, clon = cutil.add_cyclic_point(
        rescaled_meridional_mean_vertical_wind.sel(experiment=experiment).T,
        coord=meridional_mean_vertical_wind.lon,
    )

    lon_spacing = 6
    height_spacing = 3
    # scale = 0.5
    scale = 0.1
    quiv = ax.quiver(
        clon[::lon_spacing],
        meridional_mean_vertical_wind.plev[::height_spacing],
        cu[::height_spacing, ::lon_spacing],
        comega[::height_spacing, ::lon_spacing],
        scale_units='xy',
        angles='xy',
        scale=scale,
        width=0.0015,
        headwidth=6,
        headlength=10,
        minlength=2,
        zorder=10,
        color='black',
        alpha=0.7
    )
    scale_label = 1
    ax.quiverkey(quiv, X=0.9, Y=1.05, U=scale_label, label=f"{scale_label} m/s", labelpos='E')

    # Axis parameters
    ax.set_aspect("auto")

    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks, "lon", precision=0))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(100, 950)
    ax.set_yticks(np.arange(100, 1000, 100))
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    [ax.axvline(x=tick, color='#bcbcbc', alpha=0.9, ls=':') for tick in x_ticks]
    ax.axhline(y=plev_to_plot[experiment], color='#bcbcbc', alpha=0.9, ls=':')

# Save figure
output_filename = f"{contourf_variable.lower().replace(" ", "_")}_{reference_variable}_longitude_latitude"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}")
print(f"Output filename: {output_filename}/regression_analysis")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

#### Phase Shift

In [ ]:
savefig=False

longitude_bounds = {}
# longitude_bounds['-4K'] = (30, 260)
# longitude_bounds['0K'] = (30, 300)
# longitude_bounds['4K'] = (60, 300)
longitude_bounds['-4K'] = (0, 360)
longitude_bounds['0K'] = (0, 360)
longitude_bounds['4K'] = (0, 360)

first_variable = multi_experiment_variables_regressed['Precipitation']
second_variable = multi_experiment_variables_regressed['Vertical Wind']

meridional_mean_data = {}
for variable_data in [first_variable, second_variable]:

    meridional_mean_data[variable_data.name] = xr.concat(
        [
            variable_data.sel(experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])).mean(dim=['lat'])
            for experiment in coords.experiments.values
        ],
        dim=coords.experiments
    )

# level_of_maximum_moisture = {}
phase_shift = {}
delta_lon = 2.5
fontsize = 16
plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(9, 12))
gs = fig.add_gridspec(3, 1, hspace=0.3)
gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95)

axes_list = []
axes_list.append(fig.add_subplot(gs[0]))
axes_list.append(fig.add_subplot(gs[1]))
axes_list.append(fig.add_subplot(gs[2]))

for index, (ax, experiment) in enumerate(zip(axes_list, coords.experiments.values)):

    # Signals
    signal1 = standardize_data(meridional_mean_data[first_variable.name].sel(experiment=experiment), dim='lon', unit_variance=False)
    signal2 = standardize_data(meridional_mean_data[second_variable.name].sel(experiment=experiment), dim='lon', unit_variance=False)
    freqs = np.fft.fftfreq(len(signal1), d=delta_lon)

    target_wavenumber = 3
    target_freq = target_wavenumber / 360  # cycles per degree
    # FFT for dominant wavelength
    fft = np.fft.fft(signal1)
    dominant_idx = np.argmax(np.abs(fft[1:])) + 1  # skip zero freq
    dominant_fft = fft[target_wavenumber]
    filtered_fft = np.zeros_like(fft, dtype=complex)
    mask = (np.isclose(freqs, target_freq) | np.isclose(freqs, -target_freq))
    filtered_fft[mask] = fft[mask]
    dominant_signal1 = np.fft.ifft(filtered_fft).real

    #FFR 2
    fft2 = np.fft.fft(signal2)
    dominant_idx2= np.argmax(np.abs(fft2[1:])) + 1
    dominant_fft2 = fft2[target_wavenumber]
    filtered_fft2 = np.zeros_like(fft2, dtype=complex)
    mask = (np.isclose(freqs, target_freq) | np.isclose(freqs, -target_freq))
    filtered_fft2[mask] = fft2[mask]
    dominant_signal2 = np.fft.ifft(filtered_fft2).real

    phase_shift[experiment] = (
        (180/np.pi)*np.angle(dominant_fft)
        - (180/np.pi)*np.angle(dominant_fft2)
    )
    if phase_shift[experiment] <= -180:
        phase_shift[experiment] = phase_shift[experiment] + 360

    # Plot phase
    ax.set_title(
        f"{string.ascii_letters[index]}) "
        f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]} "
        f"({tick_labeller([latitude_bounds[experiment][0]], 'lat')[0]}-"
        f"{tick_labeller([latitude_bounds[experiment][1]], 'lat')[0]}): "
        fr"$\varphi$(k={target_wavenumber:0.0f})={phase_shift[experiment]:0.0f}°",
        loc='left'
)

    cdata1, clon1 = cutil.add_cyclic_point(dominant_signal1, signal1.lon)
    cdata2, clon2 = cutil.add_cyclic_point(dominant_signal2, signal2.lon)
    ax.plot(clon1, cdata1, color=bmh_colors(index+1), label=signal1.name)
    ax.plot(clon2, cdata2, color=bmh_colors(index+1), ls='--', label=signal2.name)
    ax.set_xticks(np.arange(0, 360+60, 60), labels=tick_labeller(np.arange(0, 360+60, 60), 'lon'))
    ax.set_xlim(0, 360)
    ax.legend(loc='upper left', fontsize=14)
    ax.axhline(y=0, color='gray', ls=':')
    ax.axvline(x=180, color='gray', ls=':')

output_filename = (
    f"mjo_{data_type}"
    f"_{signal1.name.lower().replace(" ", "_")}-{signal2.name.lower().replace(" ", "_")}"
    f"_k={target_wavenumber}"
    f"_phase_shift"
)
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/")
print(f"Output file: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

# print(f"Phase Angle between {first_variable.name} \n& {second_variable.name}")
# for experiment in coords.experiments.values:
#     print(f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]:<4}: {phase_shift[experiment]:>6.1f}°")

In [ ]:
savefig = False

plt.style.use('bmh')
fontsize = 16
plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(1, 2, wspace=0.3)
gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95)

axes_list = []
axes_list.append(fig.add_subplot(gs[0]))
axes_list.append(fig.add_subplot(gs[1]))

if not savefig:
    fig.suptitle(
        f"Vertically Resolved Phase Shift \n between Precipitation & Moisture",
        y=1.125
    )

for index, experiment in enumerate(coords.experiments.values):
    for ax_index, (ax, k) in enumerate(zip(axes_list, [1,2])):
        ax.set_title(f"{string.ascii_letters[ax_index]}) k={k} component", loc='left', fontsize=fontsize)
        ax.axvline(x=0, ls=':', color='#bcbcbc')
        ax.plot(
            phase_shift.sel(experiment=experiment, zonal_wavenumber=k),
            phase_shift.plev,
            color=bmh_colors(index+1),
            label=(
                f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}"
                f" ({tick_labeller(latitude_bounds[experiment], 'lat', precision=0)[0]}"
                f"-{tick_labeller(latitude_bounds[experiment], 'lat', precision=0)[1]})"
            ),
        )

        ax.set_ylim(100, 1000)
        ax.set_yticks(np.arange(100, 1100, 100))
        ax.invert_yaxis()
        ax.set_xlim(-180, 180)
        ax.set_xticks(np.arange(-180, 180+60, 60))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.grid(True)
        ax.set_ylabel('Pressure (hPa)', size=fontsize)
        ax.set_xlabel('Phase Angle (degrees)', size=fontsize)
        ax.legend(fontsize=fontsize-2)

output_filename = (
    f"mjo_{data_type}"
    f"_{signal1.name.lower().replace(" ", "_")}-{signal2.name.lower().replace(" ", "_")}"
    f"_vertically_resolved_phase_shift"
)
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/")
print(f"Output file: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

### Multiple reference variables

In [ ]:
savefig = False
statistically_significant = False

latitude_bounds = {}
latitude_bounds['-4K'] = (-5, 5)
latitude_bounds['0K'] = (-10, 10)
latitude_bounds['4K'] = (-15, 15)

fontsize = 16
plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(24, 12))
gs_main = fig.add_gridspec(1, 2, width_ratios = [45, 1], wspace=0.1)
gs_main.update(left=0.05, right=0.95, bottom=0.05, top=0.95)
gs_axes = gs_main[0].subgridspec(2, 3, hspace=0.25)
gs_cbar = gs_main[1].subgridspec(1, 1)

axes = [[],[]]
axes[0].append(fig.add_subplot(gs_axes[0,0]))
axes[0].append(fig.add_subplot(gs_axes[0,1]))
axes[0].append(fig.add_subplot(gs_axes[0,2]))

axes[1].append(fig.add_subplot(gs_axes[1,0]))
axes[1].append(fig.add_subplot(gs_axes[1,1]))
axes[1].append(fig.add_subplot(gs_axes[1,2]))

# cbar_ax = fig.add_subplot(gs[:, 1])
# cbar_axes = []
cbar_axes = fig.add_subplot(gs_cbar[0])
# cbar_axes.append(fig.add_subplot(gs_cbar[1]))

data_type = 'regressed'

contourf_variable = 'Meridional Moisture Advection'
contour_variable = 'Moisture'

if data_type == 'regressed':
     #### Contourf variable
    if 'Moisture Advection' in contourf_variable:
        ### Advection Variables
        contourf_data = 1000*config.SECONDS_PER_DAY*multi_experiment_variables_regressed[contourf_variable].where(
             (multi_experiment_significant_indices[contourf_variable] if statistically_significant else True)
        ).transpose('experiment', 'level', 'plev', 'lon', 'lat')
        contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"

    else:
        contourf_data = multi_experiment_variables_regressed[contourf_variable].where(
            (multi_experiment_significant_indices[contourf_variable] if statistically_significant else True)
        ).transpose('experiment', 'level', 'plev', 'lon', 'lat')

    #### Contour Variable
    if contour_variable != 'None':
        contour_data = 1000*multi_experiment_variables_regressed[contour_variable].where(
            (multi_experiment_significant_indices[contour_variable] if statistically_significant else True)
        ).transpose('experiment', 'level', 'plev', 'lon', 'lat')

    #### Wind Variables
    zonal_wind_data = multi_experiment_variables_regressed['Zonal Wind'].where(
        (multi_experiment_significant_indices['Zonal Wind'] if statistically_significant else True)
    ).transpose('experiment', 'level', 'plev', 'lon', 'lat')
    vertical_wind_data = multi_experiment_variables_regressed['Vertical Wind'].where(
        (multi_experiment_significant_indices['Vertical Wind'] if statistically_significant else True)
    ).transpose('experiment', 'level', 'plev', 'lon', 'lat')

elif data_type == 'correlated':
    contourf_data = multi_experiment_variables_correlated[contourf_variable].transpose('experiment', 'level', 'plev', 'lon', 'lat')
    contour_data = multi_experiment_variables_correlated[contour_variable].transpose('experiment', 'level', 'plev', 'lon', 'lat')

meridional_mean_contourf_data = xr.concat(
    [
        contourf_data.sel(
            experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])
        ).mean(dim=['lat'])
        for experiment
        in coords.experiments.values
    ],
    dim=coords.experiments
)
meridional_mean_contour_data = xr.concat(
    [
        contour_data.sel(
            experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])
        ).mean(dim=['lat'])
        for experiment
        in coords.experiments.values
    ],
    dim=coords.experiments
)

if data_type == 'regressed':
    meridional_mean_zonal_wind = xr.concat(
        [
            zonal_wind_data.sel(
                experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim=['lat'])
            for experiment
            in coords.experiments.values
        ],
        dim=coords.experiments
    )
    meridional_mean_vertical_wind = xr.concat(
        [
            vertical_wind_data.sel(
                experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim=['lat'])
            for experiment
            in coords.experiments.values
        ],
        dim=coords.experiments
    )

    # rescaled_meridional_mean_zonal_wind = meridional_mean_zonal_wind / meridional_mean_zonal_wind.std(dim=['plev', 'lon'])
    rescaled_meridional_mean_vertical_wind = meridional_mean_vertical_wind / meridional_mean_vertical_wind.std(dim=['plev', 'lon'])

    rescaled_meridional_mean_zonal_wind = meridional_mean_zonal_wind
    rescaled_meridional_mean_vertical_wind = 500*meridional_mean_vertical_wind

letter = [['a', 'b', 'c'], ['d', 'e', 'f']]


for level_index, level in enumerate(multi_experiment_variables_regressed['Moisture'].level):
    for experiment_index, experiment in enumerate(coords.experiments.values):

        #### Title
        axes[level_index][experiment_index].set_title(
            f"{letter[level_index][experiment_index]}) {config.EXPERIMENT_DISPLAY_NAMES[experiment]}"
            + f" ({tick_labeller(latitude_bounds[experiment], 'lat', precision=0)[0]}"
            + f"-{tick_labeller(latitude_bounds[experiment], 'lat', precision=0)[1]})",
            loc='left'
        )

        #### Contourf Data
        grand_max = meridional_mean_contourf_data.max()
        grand_min = meridional_mean_contourf_data.min()
        data_order = min(
            np.floor(np.log10(np.abs(grand_min))),
            np.floor(np.log10(np.abs(grand_max)))
        )

        if data_type == 'regressed':
            levels = np.arange(
                round_out(grand_min, data_order+1),
                round_out(grand_max, data_order+1)+10**data_order,
                10**data_order
            )
            index = 1
            while len(levels) < 18:
                levels = np.arange(
                round_out(grand_min, data_order+1),
                round_out(grand_max, data_order+1)+(10**data_order/2**index),
                10**data_order/2**index
                )
                index += 1
            levels -= np.min(np.abs(levels))
        else:
            magnitude_max = max(np.abs(grand_min), np.abs(grand_max))
            step=1
            increment = -2
            levels = np.arange(
                -round_out(magnitude_max, -data_order-increment),
                round_out(magnitude_max, -data_order-increment)+10**data_order,
                10**data_order
            )
            index = 1
            while len(levels) < 18:
                levels = np.arange(
                    -round_out(magnitude_max, 1, step=step),
                    round_out(magnitude_max, 1, step=step)+10**data_order/2**index,
                    10**data_order/2**index
                )
                index += 1

            # levels -= levels[np.argmin(np.abs(levels))]

        data_captured = (grand_max.values < np.max(levels))*(grand_min.values > np.min(levels))
        print(f"({level.values}, {experiment}) Colorbar captures all data? {data_captured}")

        contourf_cdata, clon = cutil.add_cyclic_point(
            meridional_mean_contourf_data.sel(experiment=experiment, level=level),
            coord=meridional_mean_contourf_data.lon,
        )

        im = axes[level_index][experiment_index].contourf(
            clon,
            meridional_mean_contourf_data.plev,
            contourf_cdata,
            cmap=modified_colormap('BrBG', 'white', 0.05, 0.05),
            levels=levels,
            norm=mcolors.CenteredNorm(vcenter=0, halfrange=max(np.abs(grand_min.values), np.abs(grand_max.values)))
        )

        cbar = fig.colorbar(
            im,
            # cax=cbar_axes[index],
            cax=cbar_axes,
            label=(meridional_mean_contourf_data.attrs['units'] if data_type == 'regressed' else ''),
            orientation="vertical",
        )
        cbar.ax.tick_params(labelsize=fontsize)
        cbar.set_ticklabels([f"{level:0.2f}" for level in im.levels])

        #### Contour Data
        contour_cdata, clon = cutil.add_cyclic_point(
            meridional_mean_contour_data.sel(experiment=experiment, level=level),
            coord=meridional_mean_contour_data.lon,
        )

        cs = axes[level_index][experiment_index].contour(
            clon,
            meridional_mean_contour_data.plev,
            contour_cdata,
            colors='black',
            levels=11,
            # levels = im.levels[::2],
            linewidths=0.5,
            alpha=0.8
        )
        axes[level_index][experiment_index].clabel(cs, inline=True, fontsize=10)

        #### Quiver Data
        if data_type == 'regressed':
            cu, clon = cutil.add_cyclic_point(
                rescaled_meridional_mean_zonal_wind.sel(experiment=experiment, level=level),
                coord=meridional_mean_zonal_wind.lon,
            )

            comega, clon = cutil.add_cyclic_point(
                rescaled_meridional_mean_vertical_wind.sel(experiment=experiment, level=level),
                coord=meridional_mean_vertical_wind.lon,
            )

            lon_spacing = 6
            height_spacing = 2
            scale = 1.5
            quiv = axes[level_index][experiment_index].quiver(
                clon[::lon_spacing],
                meridional_mean_vertical_wind.plev[::height_spacing],
                cu[::height_spacing, ::lon_spacing],
                comega[::height_spacing, ::lon_spacing],
                scale_units='xy',
                angles='xy',
                scale=scale,
                width=0.0015,
                headwidth=6,
                headlength=10,
                minlength=2,
                zorder=10,
                color='black',
                alpha=0.7
            )
            scale_label = 20
            axes[level_index][experiment_index].quiverkey(quiv, X=0.9, Y=1.05, U=scale_label, label=f"{scale_label} m/s", labelpos='E')


    #### Configure axes
    for ax in axes[level_index]:
        # Axis parameters
        ax.set_aspect("auto")

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks, "lon", precision=0))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(100, 950)
        ax.set_yticks(np.arange(100, 1000, 100))
        ax.set_ylabel("Pressure (hPa)")
        ax.invert_yaxis()
        ax.axvline(x=180, color='#bcbcbc', ls=':')

#### Set suptitle
fig.suptitle(
    (f"{contourf_data.name} (colors), {contour_data.name} (contours)" + (f" & Winds (vectors) \n " if data_type == 'regressed' else "\n")
    + f"{('regressed onto' if data_type == 'regressed' else 'correlated with')} Upper (top) & Lower level (bottom) Moisture"),
    x=(axes[0][1].get_position().x0 + axes[0][1].get_position().x1)/2,
    ha='center',
    y=1.075,
    fontsize=24
)

output_filename = (
    f"/mjo_{data_type}_"
    + f"{contourf_variable.lower().replace(" ", "_")}"
    + f"_{reference_variable}_longitude_height"
)

print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/")
print(f"Output file: {output_filename}")
#### Save output
if savefig:
    print(f"{f'Saving...':<12}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regression_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not saving")
    plt.show()

In [ ]:
# plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(24, 12))
gs_main = fig.add_gridspec(1, 2, width_ratios = [45, 1], wspace=0.1)
gs_main.update(left=0.05, right=0.95, bottom=0.05, top=0.95)
gs_axes = gs_main[0].subgridspec(2, 3, hspace=0.25)
gs_cbar = gs_main[1].subgridspec(1, 1)

axes = [[],[]]
axes[0].append(fig.add_subplot(gs_axes[0,0]))
axes[0].append(fig.add_subplot(gs_axes[0,1]))
axes[0].append(fig.add_subplot(gs_axes[0,2]))

axes[1].append(fig.add_subplot(gs_axes[1,0]))
axes[1].append(fig.add_subplot(gs_axes[1,1]))
axes[1].append(fig.add_subplot(gs_axes[1,2]))

cbar_axes = fig.add_subplot(gs_cbar[0])

contribution = config.SECONDS_PER_DAY*(
    multi_experiment_variables_regressed['Meridional Moisture Advection']
    *multi_experiment_variables_regressed['Moisture']
)

for level_index, level in enumerate(multi_experiment_variables_regressed['Moisture'].level):
    for experiment_index, experiment in enumerate(coords.experiments.values):

        contourf_cdata, clon = cutil.add_cyclic_point(
            contribution.sel(experiment=experiment, level=level, lat=slice(*coords.latitude_bounds[experiment])).mean(dim='lat'),
            coord=meridional_mean_contourf_data.lon,
        )
        grand_min = contribution.min()
        grand_max = contribution.max()
        levels = np.linspace(grand_min, grand_max, 21)

        im = axes[level_index][experiment_index].contourf(
            clon,
            contribution.plev,
            contourf_cdata,
            cmap=modified_colormap('BrBG', 'white', 0.05, 0.05),
            # levels=levels,
            levels=np.linspace(-5*10e-7, 3*10e-7, 21),
            norm=mcolors.CenteredNorm(vcenter=0)
        )

        # im = axes[level_index][experiment_index].contourf(
        #     multi_experiment_variables_regressed[process].lon,
        #     multi_experiment_variables_regressed[process].plev,
        #     contribution,
        #     levels=np.linspace(contribution.,
        #     # norm=mcolors.CenteredNorm
        # )

        fig.colorbar(im, cax=cbar_axes)

    #### Configure axes
    for ax in axes[level_index]:
        # Axis parameters
        ax.set_aspect("auto")

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks, "lon", precision=0))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(100, 950)
        ax.set_yticks(np.arange(100, 1000, 100))
        ax.set_ylabel("Pressure (hPa)")
        ax.invert_yaxis()
        ax.axvline(x=180, color='#bcbcbc', ls=':')

plt.show()

In [ ]:
maintenance_contribution = {}
# budget_propagation_contribution = {}
for process in ['Zonal Moisture Advection', 'Meridional Moisture Advection', 'Vertical Moisture Advection']:
    maintenance_contribution[process] = []
    for experiment in coords.experiments.values:
        numerator = (
            multi_experiment_variables_regressed[process].sel(experiment=experiment, level='upper', lat=slice(*coords.latitude_bounds[experiment])).mean(dim='lat')
            *multi_experiment_variables_regressed['Moisture'].sel(experiment=experiment, level='upper', lat=slice(*coords.latitude_bounds[experiment])).mean(dim='lat')
        ).integrate('plev').integrate('lon')
        denominator = (
            multi_experiment_variables_regressed['Moisture'].sel(experiment=experiment, level='upper', lat=slice(*coords.latitude_bounds[experiment])).mean(dim='lat')**2
        ).integrate('plev').integrate('lon')
        maintenance_contribution[process].append(numerator/denominator)

    maintenance_contribution[process] = xr.concat(
        [term for term in maintenance_contribution[process]],
        dim=coords.experiments
    )

In [ ]:
# projected_variable = "Moist Static Energy"

budget_term_attributes = {
    'Zonal Moisture Advection': {
        'label': r'-$\langle u \partial_{x} q \rangle$',
        'color': mcolors.LinearSegmentedColormap.from_list("HighContrastPinks", ["#fff0f5", "#ffb3cc", "#ff80b3", "#ff3385", "#cc005f"], N=256)
    },
    'Meridional Moisture Advection': {
        'label': r'-$\langle v \partial_{y} q \rangle$',
        'color': mcolors.LinearSegmentedColormap.from_list("Golden",["#fff9cc", "#ffe999", "#ffcc66",  "#e6b347", "#c49e35"], N=256),
    },
    'Vertical Moisture Advection': {
        'label': r"-$\langle \omega \partial_{p} q \rangle$",
        'color': plt.get_cmap('Purples'),
    },
}

processes_to_plot = [
    'Zonal Moisture Advection',
    'Meridional Moisture Advection',
    # 'Vertical Moisture Advection',
]

plt.style.use('bmh')
plt.rcParams.update({'font.size': 16, 'mathtext.fontset': 'dejavusans'})

fig, ax = plt.subplots(1, 1, figsize=(16, 9))
# ax.set_title(
#     (
#         f"{projection_variable} Budget Maintenance Contributions "
#     + f"\n Ressel Code, {budget_maintenance_contribution[projection_variable].attrs['Regression Method']} Regression Method"
#     ),
#     pad=15
# )

bar_width = 0.25
x_positions = np.arange(len(processes_to_plot))

# Plot bars for each experiment
for i, experiment in enumerate(coords.experiments.values):
    ax.bar(
        x_positions + i * bar_width,
        [config.SECONDS_PER_DAY*maintenance_contribution[process].sel(experiment=experiment).values for process in processes_to_plot],
        width=bar_width,
        label=config.EXPERIMENT_DISPLAY_NAMES[experiment],
        color=[budget_term_attributes[process]['color'](0.3 + 0.35 * i) for process in processes_to_plot],
        edgecolor='#bcbcbc',
        lw=2,
    )

ax.set_ylabel(r'day$^{-1}$')
ax.set_xticks(x_positions + bar_width, labels=[budget_term_attributes[process]['label'] for process in processes_to_plot], fontsize=18)
ax.axhline(y=0, color='#bcbcbc', lw=3)
ax.grid(axis='x')

for spine in ax.spines.values():
    spine.set_edgecolor("#bcbcbc")
    spine.set_linewidth(3)

# Add legend to distinguish experiments
ax.legend(fontsize=14)

plt.show()


## Latitude-Height

In [ ]:
lat_spacing = 1
height_spacing = 2
plt.style.use("default")
plt.rcParams.update({"font.size": 16})

contourf_variable = 'Meridional Moisture Advection'
contour_variable = 'Moisture'

fig = plt.figure(figsize=(16, 20))
gs = GridSpec(3, 2, width_ratios=[30, 1], height_ratios=[1, 1, 1], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.1)

axes = []
axes.append(fig.add_subplot(gs[0,0]))
axes.append(fig.add_subplot(gs[1,0]))
axes.append(fig.add_subplot(gs[2,0]))
cbar_ax = fig.add_subplot(gs[:, 1])
for index, experiment in enumerate(coords.experiments.values):

    zonal_mean_contourf_data = 1000*config.SECONDS_PER_DAY*multi_experiment_variables_regressed[contourf_variable].where(
        multi_experiment_significant_indices[contourf_variable]
    ).mean(dim='lon').T

    zonal_mean_contour_data = 1000*multi_experiment_variables_regressed[contour_variable].where(
        multi_experiment_significant_indices[contour_variable]
    ).mean(dim='lon')

    zonal_mean_zonal_wind = multi_experiment_variables_regressed['Zonal Wind'].where(
        multi_experiment_significant_indices['Zonal Wind']
    ).mean(dim='lon')
    zonal_mean_zonal_wind /= np.nanmax(np.abs(zonal_mean_zonal_wind))

    zonal_mean_vertical_wind = multi_experiment_variables_regressed['Vertical Wind'].where(
        multi_experiment_significant_indices['Vertical Wind']
    ).mean(dim='lon')
    zonal_mean_vertical_wind /= np.nanmax(np.abs(zonal_mean_vertical_wind))

    axes[index].set_title(f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}")

    im = axes[index].contourf(
        zonal_mean_contourf_data.lat,
        zonal_mean_contourf_data.plev,
        zonal_mean_contourf_data.sel(experiment=experiment).T,
        cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
        levels=21,
        norm=mcolors.CenteredNorm(vcenter=0)
    )
            # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cbar_ax,
        label=zonal_mean_variable_data.attrs['units'],
        orientation="vertical",
    )
    cbar.ax.tick_params(labelsize=20)

    cs = axes[index].contour(
        zonal_mean_contour_data.lat,
        zonal_mean_contour_data.plev,
        zonal_mean_contour_data.sel(experiment=experiment).T,
        # cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
        colors='k',
        levels=21,
        norm=mcolors.CenteredNorm(vcenter=0)
    )

    axes[index].quiver(
        zonal_mean_zonal_wind.lat[::lat_spacing],
        zonal_mean_vertical_wind.plev[::height_spacing],
        zonal_mean_zonal_wind.sel(experiment=experiment).T[::height_spacing, ::lat_spacing],
        -zonal_mean_vertical_wind.sel(experiment=experiment).T[::height_spacing, ::lat_spacing],
    )
    # axes[index].invert_yaxis()
    # Axis parameters
    axes[index].set_xlim(-30, 30)
    x_ticks = np.arange(-30, 30 + 10, 10)
    axes[index].set_xticks(x_ticks)
    # ax.set_xticklabels(tick_labeller(x_ticks, "lon"))
    # ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    axes[index].set_xlabel("Latitude")

    axes[index].set_ylim(100, 950)
    axes[index].set_yticks(np.arange(100, 1000, 100))
    axes[index].set_ylabel("Pressure (hPa)")
    axes[index].invert_yaxis()
    axes[index].set_aspect('auto')

fig.suptitle(
    'Zonal-mean MJO-regressed Moisture & Winds',
    x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
    ha='center',
    y=1.025
)

plt.show()

## Hovmoller Plots

In [ ]:
fontsize = 18
plt.rcParams.update({'font.size':fontsize})
cmap = modified_colormap('BrBG', 'white', 0.05, 0.05)
cmap.set_bad('white')
fig = plt.figure(figsize=(16, 16))
gs = GridSpec(2, 3, height_ratios=[60, 1], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.15, wspace=0.25)

axes_list = []
axes_list.append(fig.add_subplot(gs[0,0]))
axes_list.append(fig.add_subplot(gs[0,1]))
axes_list.append(fig.add_subplot(gs[0,2]))

cbar_axes = fig.add_subplot(gs[-1, :])

time_range = (365*3, 365*6)
plev_to_plot = {}
plev_to_plot['-4K'] = 850
plev_to_plot['0K'] = 850
plev_to_plot['4K'] = 850

for index, (ax, experiment) in enumerate(zip(axes_list, coords.experiments.values)):
    ax.set_title(
        f"{string.ascii_letters[index]}) "
        f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}"
        f" ({tick_labeller([latitude_bounds[experiment][0]], 'lat')[0]}-"
        f"{tick_labeller([latitude_bounds[experiment][1]], 'lat')[0]})"
        f" {plev_to_plot[experiment]} hPa",
        loc='left',
        pad=5
    )

    cdata, clon = cutil.add_cyclic_point(
        meridional_mean_moisture_anomalies[experiment].isel(time=slice(*time_range)).sel(plev=plev_to_plot[experiment]),
        coord=meridional_mean_moisture_anomalies[experiment].lon
    )

    im = ax.contourf(
        clon,
        np.arange(*time_range),
        1000*cdata,
        levels = np.linspace(-2.5, 2.5, 21),
        extend='both',
        cmap=modified_colormap('coolwarm', 'white', 0.05, 0.05)
    )
cbar = fig.colorbar(im, cax = cbar_axes, orientation='horizontal')
cbar.set_label(fr"g kg$^{{{-1}}}$")

time_labels = [
    f"{day.dt.strftime("%b").values}{day.dt.year.values:02d}"
    for day in meridional_mean_moisture_anomalies[experiment].isel(time=slice(*time_range)).time
]
for index, ax in enumerate(axes_list):
    x_ticks = np.arange(0, 360+90, 90)
    ax.set_xticks(x_ticks, labels=tick_labeller(x_ticks, 'lon'))

    ax.set_yticks(
        np.arange(*time_range)[::48],
        labels=time_labels[::48]
    )

plt.show()

In [ ]:
savefig=True
xr.set_options(keep_attrs=True)
meridional_mean_region = slice(-10,10)

for regression_PC in [1,2]:
    print(f'{f"Regressions onto PC {regression_PC}":^{config.SEP_WIDTH}}')
    print(f"{'='*config.SEP_WIDTH}")

    variables_to_plot = [
        multi_experiment_variables_regressed[regression_PC]['zonal wind'.title()],
        multi_experiment_variables_regressed[regression_PC]['meridional wind'.title()],
        multi_experiment_variables_regressed[regression_PC]['vertical wind'.title()],
        multi_experiment_variables_regressed[regression_PC]['temperature'.title()],
        multi_experiment_variables_regressed[regression_PC]['moisture'.title()],
        multi_experiment_variables_regressed[regression_PC]['geopotential height'.title()],
        multi_experiment_variables_regressed[regression_PC]['moist static energy'.title()],
        multi_experiment_variables_regressed[regression_PC]['longwave heating rate'.title()],
        multi_experiment_variables_regressed[regression_PC]['shortwave heating rate'.title()],
    ]

    for variable_data in variables_to_plot:

        zonal_mean_variable_data = variable_data.mean(dim='lon')

        grand_max = zonal_mean_variable_data.max()
        grand_min = zonal_mean_variable_data.min()

        plt.style.use("default")
        plt.rcParams.update({"font.size": 24})

        fig = plt.figure(figsize=(16, 16))
        gs = GridSpec(3, 2, width_ratios=[30, 1], height_ratios=[1, 1, 1], figure=fig)
        gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.1)

        axes = []
        axes.append(fig.add_subplot(gs[0,0]))
        axes.append(fig.add_subplot(gs[1,0]))
        axes.append(fig.add_subplot(gs[2,0]))
        cb_ax = fig.add_subplot(gs[:, 1])

        for ax, experiment in zip(axes, coords.experiments.values):

            ax.set_title(f"{experiment}")

            # Plot data
            im = ax.contourf(
                zonal_mean_variable_data.lat,
                zonal_mean_variable_data.plev,
                zonal_mean_variable_data.sel(experiment=experiment).T,
                levels=16,
                norm=mcolors.CenteredNorm(vcenter=0),
                **{k: copy.deepcopy(v) for k, v in {
                    # "norm": plotting_attributes[zonal_mean_variable_data.name].get("norm"),
                    "cmap": plotting_attributes[zonal_mean_variable_data.name].get("d_cmap")
                }.items() if v is not None}
            )

            # Add colorbar
            cbar = fig.colorbar(
                im,
                cax=cb_ax,
                label=zonal_mean_variable_data.attrs['units'],
                orientation="vertical",
            )
            cbar.ax.tick_params(labelsize=20)

            # Axis parameters
            ax.set_xlim(-30, 30)
            x_ticks = np.arange(-30, 30+10, 10)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(tick_labeller(x_ticks, "lat"))
            # ax.xaxis.set_major_locator(mticker.MaxNLocator(5, prune="lower"))
            ax.set_xlabel("Latitude")

            ax.set_ylim(100, 950)
            ax.set_yticks(np.arange(100, 1000, 100))
            ax.set_ylabel("Pressure (hPa)")
            ax.invert_yaxis()

            ax.set_aspect('auto')

        fig.suptitle(
            (
                f"Zonal-Mean"
                + f" {variable_data.name.title()} \nRegressed onto {compositing_variable.name} PC{regression_PC}"
            ),
            x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
            ha='center',
            y=1.025
        )

        if not savefig:
            plt.show()
        else:
            save_string = (
                f"multi-experiment_zonal-mean"
              + f"_{zonal_mean_variable_data.attrs['file_id']}"
              + f"_regressed-on-{compositing_variable.attrs['file_id']}-PC{regression_PC}"
              + f".png"
                )
            print(f"Saving {variable_data.name} plot as {save_string}")
            plt.savefig(
                f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regressions/latitude-height/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

print(f"{'='*40}")
print("Finished")

## Lag Regressions

### Lag regress variables

In [ ]:
# savefig = False

# PCs_to_use = [1, 2]

# correlation = []
# lag_days = np.arange(-30, 31)
# for lag in lag_days:
#     correlation.append(
#         np.correlate(
#             standardize_data(PC[PCs_to_use[0]-1]),
#             np.roll(standardize_data(PC[PCs_to_use[1]-1]), shift=lag),
#         )
#         / len(PC[0])
#     )

# plt.style.use("bmh")
# plt.rcParams.update({"font.size": 20})

# [fig, ax] = plt.subplots(figsize=(16, 9))
# ax.set_title(
#     f"Lag Correlation between {experiment} {compositing_variable.name} PCs {PCs_to_use[0]} & {PCs_to_use[1]}",
#     pad=15
# )

# ax.plot(lag_days, correlation, marker="o")
# ax.axhline(y=0, color="#bcbcbc", lw=3, zorder=-10)
# ax.set_ylim(-0.8, 0.8)
# ax.set_xlim(-30, 30)
# ax.set_yticks(np.arange(-0.8, 1, 0.2))
# ax.set_xticks(np.arange(-30, 35, 5))
# ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=13, prune="lower"))
# ax.set_xlabel("Lag (day)")
# ax.set_ylabel("Correlation")
# for spine in ax.spines.values():
#     spine.set_edgecolor("#bcbcbc")
#     spine.set_linewidth(3)

# if not savefig:
#     plt.show()
# else:
#     save_string = (
#         f"{experiment}"
#       + f"_{compositing_variable.attrs['file_id']}"
#       + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
#       + f"_PC-{PCs_to_use[0]}-{PCs_to_use[1]}_lag-correlation.png"
#         )
#     print(f"Saving plot as {save_string}")
#     plt.savefig(
#         f"{output_directory}/correlations/PC-lag-correlations/{save_string}",
#         dpi=500,
#         bbox_inches="tight",
#     )

# print(f"{'='*40}")
# print("Finished")

In [ ]:
print(f"{'Lag Regressions':^{config.SEP_WIDTH}}")
# print(f"{f'Regression Variable: {regression_variable}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

meridional_mean_region = slice(-10,10)
weights = np.cos(np.deg2rad(multi_experiment_variables_mjo_filtered['Precipitation'].lat))
weights.name = "weights"
lag_days = np.arange(-30, 31)
multi_experiment_lag_regression = {}
lag_regression_variable = 'Precipitation'

# Identify reference timeseries
reference_longitude = 180
# reference_latitude = (
#     multi_experiment_variables_mjo_filtered[lag_regression_variable]
# ).var(dim='time').mean(dim='lon').idxmax(dim='lat')
reference_latitude = 0

# Select the timeseries at the correct reference location for each experiment
# selected_timeseries = []
# for exp, lat in zip(multi_experiment_variables_mjo_filtered[lag_regression_variable].experiment, reference_latitude):

#     experiment_reference_timeseries = multi_experiment_variables_mjo_filtered[lag_regression_variable].sel(
#         experiment=exp, lat=lat, lon=reference_longitude
#     )
#     selected_timeseries.append(experiment_reference_timeseries)

# Combine all timeseries into a single DataArray if needed
# reference_timeseries = xr.concat(selected_timeseries, dim="experiment")
reference_timeseries = multi_experiment_variables_mjo_filtered['Precipitation'].sel(lat=0, lon=180, method='nearest')
reference_timeseries = reference_timeseries.drop_sel(time=coords.missing_days, errors='ignore')

for index, (variable_name, variable_data) in enumerate(multi_experiment_variables_subset.items()):
    print(f'{f"({index+1}/{len(multi_experiment_variables_subset)}) {variable_name}...":<{config.SEP_WIDTH-1}}', end="")

    if not variable_name in variables_to_load:
        print(f"{variable_name} not in variables to load, skipping...")
        continue

    if 'plev' in variable_data.coords:
        multi_experiment_lag_regression[variable_name] = xr.DataArray(
            data=np.empty(
                (
                    len(coords.experiments),
                    len(lag_days),
                    len(coords.latitudes),
                    len(coords.longitudes),
                    len(coords.pressure_levels)
                )
            ),
            dims=["experiment", "lag", "lat", "lon", "plev"],
            coords=dict(
                experiment=coords.experiments,
                lag=lag_days,
                lat=coords.latitude,
                lon=coords.longitude,
                plev=coords.pressure_levels
            ),
            attrs = variable_data.attrs
        )
    else:
        multi_experiment_lag_regression[variable_name] = xr.DataArray(
            data=np.empty((len(coords.experiments), len(lag_days), len(coords.latitudes), len(coords.longitudes))),
            dims=["experiment", "lag", "lat", "lon"],
            coords=dict(
                experiment=coords.experiments,
                lag=lag_days,
                lat=coords.latitude,
                lon=coords.longitude
            ),
            attrs = variable_data.attrs
        )

    for lag_index, lag in enumerate(lag_days):
        multi_experiment_lag_regression[variable_name][:, lag_index] = np.einsum(
                'ij...,ij...->i...',
                standardize_data(variable_data, dim='time', unit_variance=False),
                np.roll(standardize_data(reference_timeseries), shift=lag),
            ) / len(reference_timeseries.time)
    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
# print(variable_data.shape)
# print(reference_timeseries.shape)

meridional_mean_region = slice(-10,10)
weights = np.cos(np.deg2rad(multi_experiment_variables_mjo_filtered['Precipitation'].lat))
weights.name = "weights"
lag_days = np.arange(-30, 31)
multi_experiment_lag_regression = {}
lag_regression_variable = 'Precipitation'
reference_timeseries = multi_experiment_variables_mjo_filtered['Precipitation'].sel(lat=0, lon=180, method='nearest')

latitude_bounds = {}
latitude_bounds['-4K'] = (-5,5)
latitude_bounds['0K'] = (-10,10)
latitude_bounds['4K'] = (-15,15)

lag_regressed_data = {}
for variable_name, variable_data in multi_experiment_variables_subset.items():
    meridional_mean_variable_data = xr.concat(
        [
            variable_data.sel(experiment=experiment, lat=slice(*coords.latitude_bounds[experiment])).mean(dim='lat')
            for experiment in coords.experiments.values
        ],
        dim=coords.experiments
    )
    standardized_data = standardize_data(meridional_mean_variable_data, dim='time', unit_variance=False)
    standardized_reference_timeseries = standardize_data(reference_timeseries, dim='time', unit_variance=False)
    lags = np.arange(-31, 31, 1)

    lag_regressed_data[variable_name] = xr.concat(
        [
            xr.dot(standardized_data, standardized_reference_timeseries.roll(time = i), dim='time') / len(reference_timeseries.time)
            for i in lags
        ],
        dim=xr.DataArray(
            data=lags,
            dims='lag',
            coords={'lag':lags}
        )
    )

In [ ]:
plt.style.use("default")
plt.rcParams.update({"font.size": 24})

fig = plt.figure(figsize=(16, 16))
gs = GridSpec(3, 1,figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.1)
axes = []
axes.append(fig.add_subplot(gs[0]))
axes.append(fig.add_subplot(gs[1]))
axes.append(fig.add_subplot(gs[2]))

for ax, experiment in zip(axes, coords.experiments.values):
    lag_regressed_data['Vertical Wind'].sel(experiment=experiment, lon=slice(150, 210)).mean(dim='lon').plot.contourf(y='plev', ax=ax, yincrease=False, levels=21)
plt.show()

In [ ]:
data_type = 'regressed'
contourf_variable = 'Precipitation'
# contourf_variable = 'Column Water Vapor'
# contour_variable = 'Moisture'

contour_variable = 'Precipitation'
# contour_variable = 'Meridional Moisture Advection'

# Scale variables to be plotted
contourf_scale = 1
contour_scale = 1
if data_type == 'regressed':
    if 'Moisture Advection' in contourf_variable:
        contourf_scale = 1000*config.SECONDS_PER_DAY
        # contourf_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contourf_variable:
        contourf_scale = 1000

    if 'Moisture Advection' in contour_variable:
        contour_scale = 1000*config.SECONDS_PER_DAY
        # contour_data.attrs['units'] = r"g kg$^{-1}$ day$^{-1}$"
    elif 'Moisture' in contour_variable:
        contour_scale = 1000

contourf_data = contourf_scale*lag_regressed_data[contourf_variable].sel(
    **({'plev': [plev for plev in plev_to_plot.values()]} if 'plev' in lag_regressed_data[contourf_variable].coords else {})
)

contour_data = contour_scale*lag_regressed_data[contour_variable].sel(
    **({'plev': [plev for plev in plev_to_plot.values()]} if 'plev' in lag_regressed_data[contour_variable].coords else {})
)


fig = plt.figure(figsize=(16, 16))
gs = GridSpec(3, 2, width_ratios=[45, 1], height_ratios=[1, 1, 1], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.1)

axes = []
axes.append(fig.add_subplot(gs[0,0]))
axes.append(fig.add_subplot(gs[1,0]))
axes.append(fig.add_subplot(gs[2,0]))
cbar_axes = []
cbar_axes.append(fig.add_subplot(gs[0, 1]))
cbar_axes.append(fig.add_subplot(gs[1, 1]))
cbar_axes.append(fig.add_subplot(gs[2, 1]))

for index, (ax, experiment) in enumerate(zip(axes, coords.experiments.values)):
    ax.set_title(f"{string.ascii_letters[index]}) {config.EXPERIMENT_DISPLAY_NAMES[experiment]}", loc='left')

    cyclic_contourf_data, cyclic_contourf_lon = cutil.add_cyclic_point(
        contourf_data.sel(
            experiment=experiment,
            # plev=850,
        ),
        coord=lag_regressed_data[contourf_variable].lon
    )

    im = ax.contourf(
        cyclic_contourf_lon,
        lag_regressed_data[contourf_variable].lag,
        cyclic_contourf_data,
        # levels=np.linspace(-2, 2, 21),
        levels=21,
        extend='both',
        cmap=modified_colormap('BrBG', 'white', 0.1, 0.05),
        norm=mcolors.CenteredNorm(vcenter=0)
    )
    cbar = fig.colorbar(im, cax=cbar_axes[index])
    # cbar.set_label(r'mm day$^{-1}$')
    cbar.set_label(f"{contourf_data.attrs['units']}")

    cyclic_contour_data, cyclic_contour_lon = cutil.add_cyclic_point(
        contour_data.sel(
            experiment=experiment,
        ),
        coord = lag_regressed_data[contour_variable].lon
    )

    cs = ax.contour(
        cyclic_contour_lon,
        lag_regressed_data[contour_variable].lag,
        cyclic_contour_data,
        colors='k',
        alpha=0.5,
        levels=7
    )
    if contour_variable != contourf_variable:
        ax.clabel(cs, inline=True, fontsize=10)

    ax.axvline(x=180, ls='--', color='gray')
    ax.axhline(y=0, ls='--', color='gray')

    ax.set_yticks(np.arange(-30, 40, 10))
    ax.set_ylabel('Lag (days)')
    ax.set_xticks(np.arange(0, 360+60, 60), labels=(tick_labeller(np.arange(0, 360+60, 60), 'lon')))

plt.show()

### Plot lag-regressed variables

#### Lag-longitude

In [ ]:
scale_factor = {}
for experiment in coords.experiments.values:
    scale_factor[experiment] = (15/multi_experiment_lag_regression['Precipitation'].sel(lon=0, lag=0, method='nearest').sel(experiment=experiment, lat=slice(*coords.latitude_bounds[experiment]))).mean(dim='lat')

In [ ]:
savefig=False

regression_variable = 'Precipitation'
reference_latitude = 0
plt.style.use("default")
plt.rcParams.update({"font.size": 24})
multi_experiment_lag_regression['Precipitation'].name = 'Precipitation'
# multi_experiment_lag_regression['Precipitation'] *= -(15/multi_experiment_lag_regression['Precipitation'].sel(lat=0, lon=0, lag=0, method='nearest'))
variables_to_plot = [
    multi_experiment_lag_regression['Precipitation'],
    # scaled_multi_experiment_lag_regression['outgoing longwave radiation'.title()],
    # scaled_multi_experiment_lag_regression['zonal wind'.title()].sel(plev=200),
    # scaled_multi_experiment_lag_regression['zonal wind'.title()].sel(plev=850),
    # multi_experiment_lag_regression['vertical wind'.title()].sel(plev=500),
    # multi_experiment_variables_regressed['column temperature'.title()],
    # multi_experiment_variables_regressed['column water vapor'.title()],
    # multi_experiment_lag_regression['moist static energy'.title()].sel(plev=850),
    # multi_experiment_variables_regressed['column longwave heating'.title()],
    # multi_experiment_variables_regressed['column shortwave heating'.title()],
    # multi_experiment_lag_regression['latent heat flux'.title()],
    # multi_experiment_lag_regression['sensible heat flux'.title()],
]

print(f"{'='*config.SEP_WIDTH}")
for variable_data in variables_to_plot:
    grand_max = variable_data.max()
    grand_min = variable_data.min()

    fig = plt.figure(figsize=(16, 16))
    gs = GridSpec(3, 2, width_ratios=[30,1], height_ratios=[1,1,1], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.4, wspace=0.1)

    axes = []
    axes.append(fig.add_subplot(gs[0,0]))
    axes.append(fig.add_subplot(gs[1,0]))
    axes.append(fig.add_subplot(gs[2,0]))
    cb_ax = fig.add_subplot(gs[:, 1])

    for ax, experiment in zip(axes, coords.experiments.values):
        ax.set_title(f"{experiment}")

        # Add cyclic point
        cdata, clon = cutil.add_cyclic_point(
            # variable_data.sel(experiment=experiment, lat=slice(-10,10)).weighted(weights).mean(dim='lat'),
            scale_factor[experiment]*variable_data.sel(
                experiment=experiment,
                # lat=reference_latitude.sel(experiment=experiment)
                lat=slice(*coords.latitude_bounds[experiment])
            ).mean(dim='lat'),
            coord=variable_data.lon,
        )

        # Plot data
        im = ax.contourf(
            clon,
            variable_data.lag,
            cdata,
            # levels=np.linspace(grand_min, grand_max, 21),
            levels=21,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                # "norm": plotting_attributes[variable_data.name].get('norm'),
                "cmap": plotting_attributes[variable_data.name].get('d_cmap')
            }.items() if v is not None}
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=variable_data.attrs['units'],
            orientation="vertical",
        )
        cbar.ax.tick_params(labelsize=20)

        # ax.plot(reference_longitude, reference_latitude.sel(experiment=experiment), marker='x', color='k', ms=14)
        ax.axhline(y=0, ls='-', color="grey", lw=1)
        ax.axvline(x=180, ls='-', color="grey", lw=1)

        # Axis parameters
        ax.set_aspect("auto")

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(-30, 30)
        y_ticks = np.arange(-30, 40, 10)
        ax.set_yticks(y_ticks)
        ax.set_ylabel("Lag (day)")

        # grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "grey"}
        ax.grid(True, **grid_kwargs)

    fig.suptitle(
        (
            f"{(' ' + (str(variable_data.plev.values) + '-hPa') if 'plev' in variable_data.coords else '')}"
            + f" {variable_data.name} lag regressed onto \n{regression_variable}{(regression_index+1 if regression_variable == 'PCs' else '')}"
        ),
        x=(axes[0].get_position().x0 + axes[0].get_position().x1)/2,
        ha='center',
        y=1.05
    )

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"multi-experiment_{variable_data.attrs['file_id']}"
          + f"{((str(variable_data.plev.values)) if 'plev' in variable_data.coords else '')}"
          + f"lag_regressed-on-{regression_variable.replace(' ', '-')}{(regression_index+1 if regression_variable == 'PCs' else '')}"
          + f".png"
            )
        print(
            f"Saving {(' ' + (str(variable_data.plev.values) + '-hPa') if 'plev' in variable_data.coords else '')} {variable_data.name} plot as {save_string}"
        )
        plt.savefig(
            f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/regressions/latitude-longitude/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

#### Lag-Latitude

In [ ]:
calculate_corr = True
plot_corr = True
savefig = False

lag_days = np.arange(-30, 31)
reference_lon = 180
meridional_mean_region = slice(-10, 10)
weights = np.cos(np.deg2rad(latitude))
weights.name = "weights"

variables_to_plot = [
    variables_filtered['precipitation'.title()],
    variables_filtered['outgoing longwave radiation'.title()],
    variables_filtered['zonal wind'.title()].sel(plev=200),
    variables_filtered['zonal wind'.title()].sel(plev=850),
    variables_filtered['vertical wind'.title()].sel(plev=500),
    variables_filtered['column temperature'.title()],
    variables_filtered['column water vapor'.title()],
]

if calculate_corr or 'lag_latitude_correlation' not in globals():
    lag_latitude_correlation = {}

for variable in variables_to_plot:
    print(f"Variable: {variable.name}")

    if calculate_corr:
        # lag_latitude_correlation[variable.name] = np.empty((len(lag_days), len(longitude)))
        lag_latitude_correlation[variable.name] = xr.DataArray(
            data=np.empty((len(lag_days), len(coords.latitude))),
            dims=["lag", "lat"],
            coords=dict(
                lag=lag_days,
                lat=coords.latitude
            )
        )

        # reference_time_series = variable.sel(lon=reference_lon, lat=meridional_mean_region).weighted(weights).mean(dim='lat')
        reference_time_series = PC[0]

        print("Computing correlations...")
        for lag_index, lag in enumerate(lag_days):
            lag_latitude_correlation[variable.name][lag_index] = np.einsum(
                    'ij,i->j',
                    standardize_data(variable.sel(lon=reference_lon)),
                    np.roll(standardize_data(reference_time_series), shift=lag),
                ) / len(reference_time_series)
        print("Correlations computed")

    if plot_corr:
        plt.style.use("bmh")
        plt.rcParams.update({"font.size": 20})
        [fig, ax] = plt.subplots(figsize=(16, 6))

        ax.set_title(
            f"{variable.name} Lag-Latitude Correlation",
            pad=10
        )

        levels = np.linspace(-1, 1, 21)

        im = ax.contourf(
            lag_latitude_correlation[variable.name].lat,
            lag_latitude_correlation[variable.name].lag,
            lag_latitude_correlation[variable.name],
            cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
            levels=levels[levels != 0]
        )
        cbar = fig.colorbar(im)
        cbar.set_ticks(np.arange(-1, 1.2, 0.2))

        ax.contour(
            lag_latitude_correlation[variable.name].lat,
            lag_latitude_correlation[variable.name].lag,
            lag_latitude_correlation[variable.name],
            colors="k",
            levels=levels[levels != 0]
        )


        ax.set_xlim(-30, 30)
        ax.set_ylim(-30, 30)
        xtick_locations = np.arange(-30, 30 + 10, 10)
        ax.set_xticks(ticks=xtick_locations, labels=tick_labeller(xtick_locations, "lat"))
        # ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_xlabel("Latitude")
        ax.set_ylabel("Lag (Day)")

        for spine in ax.spines.values():
            spine.set_edgecolor("#bcbcbc")
            spine.set_linewidth(3)

        # plt.show()
        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}"
                + f"_{compositing_variable.attrs['file_id']}"
                + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
                + f"_PC-{PCs_to_use[0]+1}-{PCs_to_use[1]+1}_lag-longitude-correlation.png"
            )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/correlations/lag-latitude/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

print(f"{'='*40}")
print("Finished")

#### Lag-Height

In [ ]:
calculate_corr = True
plot_corr = True
savefig = False

lag_days = np.arange(-30, 31)
reference_lon = 180
meridional_mean_region = slice(-15, 15)
weights = np.cos(np.deg2rad(latitude))
weights.name = "weights"

variables_to_plot = [
    variables_filtered['zonal wind'.title()],
    variables_filtered['meridional wind'.title()],
    variables_filtered['vertical wind'.title()],
    variables_filtered['temperature'.title()],
    variables_filtered['moisture'.title()],
    variables_filtered['geopotential height'.title()],
    variables_filtered['moist static energy'.title()],
    variables_filtered['longwave heating rate'.title()],
    variables_filtered['shortwave heating rate'.title()],
]

if calculate_corr or 'lag_height_correlation' not in globals():
    lag_height_correlation = {}

for variable in variables_to_plot:
    print(f"Variable: {variable.name}")

    if calculate_corr:
        lag_height_correlation[variable.name] = xr.DataArray(
            data=np.empty((len(lag_days), len(coords.pressure_levels))),
            dims=["lag", "plev"],
            coords=dict(
                lag=lag_days,
                plev=coords.pressure_levels
            )
        )

        # reference_time_series = variable.sel(lon=reference_lon, lat=meridional_mean_region, plev=500).weighted(weights).mean(dim='lat')
        reference_time_series = variables_filtered['Precipitation'].sel(
            lon=reference_lon, lat=meridional_mean_region
        ).weighted(weights).mean(dim='lat')
        # reference_time_series = PC[0]

        print("Computing correlations...")
        for lag_index, lag in enumerate(lag_days):
            lag_height_correlation[variable.name][lag_index] = np.einsum(
                    'ij,i->j',
                    # standardize_data(variable.sel(lon=reference_lon, lat=meridional_mean_region).weighted(weights).mean(dim='lat')),
                    variable.sel(lon=reference_lon, lat=meridional_mean_region).weighted(weights).mean(dim='lat'),
                    np.roll(standardize_data(reference_time_series), shift=lag),
                ) / len(reference_time_series)
        print("Correlations computed")

    if plot_corr:
        plt.style.use("bmh")
        plt.rcParams.update({"font.size": 20})
        [fig, ax] = plt.subplots(figsize=(16, 6))

        ax.set_title(
            f"{variable.name} Lag-Height Correlation",
            pad=10
        )

        # levels = np.linspace(-1, 1, 21)
        levels = np.linspace(
            lag_height_correlation[variable.name].min(),
            lag_height_correlation[variable.name].max(),
            21
        )

        im = ax.contourf(
            lag_height_correlation[variable.name].lag,
            lag_height_correlation[variable.name].plev,
            lag_height_correlation[variable.name].T,
            cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
            norm=mcolors.CenteredNorm(vcenter=0),
            levels=levels[levels != 0]
        )
        cbar = fig.colorbar(im)
        cbar.set_label(f"{variable.attrs['units']}")
        # cbar.set_ticks(np.arange(-1, 1.2, 0.2))

        # ax.contour(
        #     lag_height_correlation[variable.name].lag,
        #     lag_height_correlation[variable.name].plev,
        #     lag_height_correlation[variable.name].T,
        #     colors="k",
        #     levels=levels[levels != 0]
        # )

        ax.set_xlim(-30, 30)
        ax.set_xticks(np.arange(-30, 35, 5))
        ax.invert_xaxis()
        # ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_xlabel("Lag (Day)")

        ax.set_ylim(100, 950)
        ax.set_yticks(np.arange(100, 1000, 100))
        ax.set_ylabel("Pressure (hPa)")
        ax.invert_yaxis()

        for spine in ax.spines.values():
            spine.set_edgecolor("#bcbcbc")
            spine.set_linewidth(3)

        # plt.show()
        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}"
                + f"_{compositing_variable.attrs['file_id']}"
                + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
                + f"_PC-{PCs_to_use[0]+1}-{PCs_to_use[1]+1}_lag-longitude-correlation.png"
            )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/correlations/lag-longitude/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

print(f"{'='*40}")
print("Finished")